# Layer 1 Experiment — Maltrail (rule-based) + Zeek (flow monitoring)

**Part of:** the research project "Scalability of Orchestration in a Security Operation Center" (Kemdiktisaintek; originally targeted at ACIG)
**Layer tested:** Layer 1 of the 3-layer architecture — Maltrail and flow monitoring running **in parallel** at the same port-mirroring point.

**Specific goals of this experiment:**
1. Run **Maltrail** on labelled pcaps (malware & benign) to see its detection pattern — the basis for calibrating the "Maltrail alert history" as a Layer 2 feature (not a pass/fail gate, see the *cold-start contamination* resolution in the project notes).
2. Run **Zeek** on the same pcaps to produce `conn.log` — the baseline flow records that become the Layer 2 input (behaviour profiling/clustering).
3. Produce one summary table per pcap (ground truth vs Maltrail output vs Zeek output) as the basis for the next design decisions (aggregation window, 6 candidate features, etc. — see the end of the notebook).

**Scale:** this is a **PILOT** experiment (5 pcaps: 3 malware + 2 benign), not the whole dataset corpus. The pattern that already proved successful in Experiment 1 (ATT&CK→Neo4j) is: verify small first until it really matches, then scale — do not jump to hundreds of pcaps before this pilot pipeline is shown to run and give sensible results.

> **Note for readers of the public repository:** this notebook is stored **without outputs** because it was edited after its last full run. The executed record of the 28 August 2026 run is summarised in `results/RESULTS.md`; the alert logs it produced are in `data/maltrail_out/`.


## ⚠️ Read first — verification status of each component

Update 24 August 2026: the first draft of this notebook (see the old version if needed) kept a "guided discovery" session for Maltrail because its CLI flags were not yet known. That session is finished — it was carried out directly in a terminal on the authors' Mac, its results were verified empirically (not guessed), and they are now permanently embedded in the code of section C below. Status summary per component:

| Component | Status | Confidence |
|---|---|---|
| URL & byte size of the USTC-TFC2016 pcaps (Zeus, Tinba, BitTorrent, Skype, Cridex.7z, Shifu.7z) | Verified via `curl` HEAD (exact `content-length`) AND via a real download on the Mac — exact match | [HIGH CONFIDENCE] |
| URL of the Stratosphere Dridex pcap (`CTU-Malware-Capture-Botnet-153-1`) | Verified to exist & downloaded (17,216,694 bytes, consistent with the "~16M" estimate in the listing) | [HIGH CONFIDENCE] |
| **Maltrail sensor used: `old/sensor.py`, NOT `sensor.py`** | The Maltrail repo structure changed — the official sensor is now rewritten in Rust (`sensor/`), BUT the Rust sensor **fails to build on macOS arm64** (Linux-only `AF_PACKET` code not `cfg`-gated in `fanout.rs`, confirmed via `cargo build --release` directly). That is why this notebook uses `old/sensor.py` — the older Python version still officially maintained upstream as a differential oracle (39/39 parity tests with the Rust sensor) | [HIGH CONFIDENCE, confirmed empirically] |
| CLI flags of `old/sensor.py`: `-r <pcap> -c <conf> --console [--offline]` | Confirmed directly from `python3 old/sensor.py --help` on the Mac | [HIGH CONFIDENCE, EMPIRICAL] |
| `DISABLE_CHECK_SUDO true` (skips the root requirement for `-r` mode) | Pcap replay turned out to still require root by default (an unconditional legacy check in `core/log.py`, not a real technical need) — the solution is this official config option, whose format was confirmed from `core/testing.py` (the upstream test suite itself). `sudo` is DELIBERATELY not used, for least privilege | [HIGH CONFIDENCE, EMPIRICAL] |
| Location of Maltrail alert logs (`LOG_DIR/<YYYY-MM-DD>.log`) & line format | Confirmed from real runs — see the parser regex in section C.4, including the special case of quoted observable fields (e.g. User-Agent strings containing spaces) | [HIGH CONFIDENCE, EMPIRICAL] |
| Maltrail's main trail cache | `~/.maltrail/trails.csv` (NOT `CUSTOM_TRAILS_DIR`), containing **38 external feeds + (custom) + (static) = 40 sources**, NOT "57" as noted in an early draft of this research — no source for the number 57 could be found; corrected | [HIGH CONFIDENCE for 38+2=40, counted directly from the output; NO basis for the number 57] |
| `meta.sqlite` (`observables` table) | **NOT an alert counter** — a general traffic-observable cache (binary columns, not a "malicious" representation). This notebook does NOT use it; the source of truth for alerts is the `.log` text files | [MEDIUM-HIGH CONFIDENCE for this interpretation; the `core/` source has not been read directly] |
| pcapy-ng can be built natively on Apple Silicon (M1 Pro) | **Yes, verified** — installed from a prebuilt wheel, no compilation | [HIGH CONFIDENCE, EMPIRICAL] |
| `zeek -C -r <pcap> local` syntax for offline pcap analysis | Standard Zeek idiom | [HIGH CONFIDENCE] |
| Zeek `conn.log` TSV format (`#fields`/`#types` header) | Standard Zeek format, very stable across versions | [HIGH CONFIDENCE] |
| Stratosphere label `Botnet-224-1` = Zbot | The source itself writes **"Zbot? Gootkit?"** — uncertain | [LOW CONFIDENCE according to the source itself — do not use as "Zbot" ground truth without qualification] |
| Ursnif pcap | **No verified source found yet** — empty placeholder in the manifest | no data |
| Pilot result: Dridex (malware) = 0 Maltrail alerts | Genuinely 0 lines in the log, not a bug — unverified hypothesis: IOC staleness (2016 pcap vs 2026 blacklist snapshot) | [the "0 alerts" observation itself is HIGH; its cause (staleness) is MEDIUM and needs further verification] |

If a new assumption turns out to be wrong after re-running, that is **not a failure** — report it back so it can be fixed (the same pattern as Experiment 1).


In [ ]:
# --- Environment check ---
# Note: the "X | None" syntax is deliberately NOT used in type annotations so that this notebook
# stays safe to run across Python versions (e.g. if it turns out to run in an old 3.9 venv) —
# "from __future__ import annotations" is not reliable across cells in Jupyter, so it is avoided.
from typing import Optional
import sys, platform, shutil, subprocess

print("Python       :", sys.version)
print("Platform     :", platform.platform())
print("Architecture :", platform.machine())
print()
print("Recommendation: use a NEW venv dedicated to this experiment (not an old 3.9 EOL venv),")
print("so that native dependencies (pcapy-ng, py7zr) do not pollute another research venv.")
print()

for tool in ["zeek", "python3", "pip3", "7z", "7zz"]:
    path = shutil.which(tool)
    print(f"{tool:8s} -> {'FOUND: ' + path if path else 'NOT FOUND on PATH'}")


## Section A — Working folder structure

All experiment outputs are stored inside this folder (`layer1-maltrail-zeek/`), next to the existing `layer3-attck-neo4j/`.


In [ ]:
from pathlib import Path

BASE_DIR = Path.cwd()  # run this notebook from inside the layer1-maltrail-zeek/ folder
DATA_DIR = BASE_DIR / "data"
PCAP_RAW_DIR = DATA_DIR / "pcap_raw"
PCAP_EXTRACTED_DIR = DATA_DIR / "pcap_extracted"
MALTRAIL_OUT_DIR = DATA_DIR / "maltrail_out"
ZEEK_OUT_DIR = DATA_DIR / "zeek_out"
RESULTS_DIR = BASE_DIR / "results"

for d in [PCAP_RAW_DIR, PCAP_EXTRACTED_DIR, MALTRAIL_OUT_DIR, ZEEK_OUT_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)
    print("OK:", d)


## Section B — Pilot dataset manifest & download

Pilot set: **3 malware pcaps** (Zeus, Tinba, Dridex — all genuinely credential-stealing/banking trojans, converging with this project's research findings) + **2 benign pcaps** (BitTorrent, Skype) as controls.

`Cridex` and `Shifu` are deliberately **kept separate as OPTIONAL** (section B.2) because they come as `.7z` (extraction needed) — only run after this 5-file pilot pipeline is shown to work.

`expected_bytes` for the USTC-TFC2016 entries is the exact byte `content-length` checked directly — used as an automatic integrity check. For the Stratosphere entries the size is only an estimate from the directory listing ("~16 MB"), so the check is only "plausible order of magnitude", not byte-exact.


In [ ]:
import requests
from dataclasses import dataclass, field

@dataclass
class DatasetEntry:
    key: str
    family: str
    ground_truth: str            # "malware" or "benign"
    url: Optional[str]
    local_name: str
    expected_bytes: Optional[int]   # None = no byte-exact check
    needs_extraction: bool = False
    confidence_note: str = ""
    source_dataset: str = "USTC-TFC2016/Stratosphere"   # added 25 August 2026 -- see EXTENDED_MANIFEST

PILOT_MANIFEST = [
    DatasetEntry(
        key="zeus", family="Zeus (banking trojan)", ground_truth="malware",
        url="https://raw.githubusercontent.com/yungshenglu/USTC-TFC2016/master/Malware/Zeus.pcap",
        local_name="zeus.pcap", expected_bytes=14094482,
        confidence_note="HIGH CONFIDENCE — byte-exact, verified",
    ),
    DatasetEntry(
        key="tinba", family="Tinba (banking trojan)", ground_truth="malware",
        url="https://raw.githubusercontent.com/yungshenglu/USTC-TFC2016/master/Malware/Tinba.pcap",
        local_name="tinba.pcap", expected_bytes=2680243,
        confidence_note="HIGH CONFIDENCE — byte-exact, verified",
    ),
    DatasetEntry(
        key="dridex", family="Dridex (banking trojan / credential theft)", ground_truth="malware",
        url="https://mcfp.felk.cvut.cz/publicDatasets/CTU-Malware-Capture-Botnet-153-1/2016-02-12-capture_win4.pcap",
        local_name="dridex.pcap", expected_bytes=None,
        confidence_note="MEDIUM-HIGH CONFIDENCE — clean label at the source, size only estimated at ~16MB",
    ),
    DatasetEntry(
        key="bittorrent", family="BitTorrent (benign)", ground_truth="benign",
        url="https://raw.githubusercontent.com/yungshenglu/USTC-TFC2016/master/Benign/BitTorrent.pcap",
        local_name="bittorrent.pcap", expected_bytes=7686388,
        confidence_note="HIGH CONFIDENCE — byte-exact, verified",
    ),
    DatasetEntry(
        key="skype", family="Skype (benign)", ground_truth="benign",
        url="https://raw.githubusercontent.com/yungshenglu/USTC-TFC2016/master/Benign/Skype.pcap",
        local_name="skype.pcap", expected_bytes=4433982,
        confidence_note="HIGH CONFIDENCE — byte-exact, verified",
    ),
]

# --- OPTIONAL, next scaling round (not part of the 5-file pilot) ---
OPTIONAL_MANIFEST = [
    DatasetEntry(
        key="cridex", family="Cridex (Dridex variant)", ground_truth="malware",
        url="https://raw.githubusercontent.com/yungshenglu/USTC-TFC2016/master/Malware/Cridex.7z",
        local_name="cridex.7z", expected_bytes=9463176, needs_extraction=True,
        confidence_note="HIGH CONFIDENCE download, needs 7z to open",
    ),
    DatasetEntry(
        key="shifu", family="Shifu (banking trojan)", ground_truth="malware",
        url="https://raw.githubusercontent.com/yungshenglu/USTC-TFC2016/master/Malware/Shifu.7z",
        local_name="shifu.7z", expected_bytes=4416510, needs_extraction=True,
        confidence_note="HIGH CONFIDENCE download, needs 7z to open",
    ),
    DatasetEntry(
        key="zbot_gootkit", family="Zbot OR Gootkit (label UNCERTAIN according to the source)", ground_truth="malware",
        url="https://mcfp.felk.cvut.cz/publicDatasets/CTU-Malware-Capture-Botnet-224-1/capture_win11.pcap",
        local_name="zbot_or_gootkit.pcap", expected_bytes=None,
        confidence_note="LOW CONFIDENCE label — the source itself writes 'Zbot? Gootkit?'. Do not use as ground truth without qualification.",
    ),
    DatasetEntry(
        key="ursnif", family="Ursnif — NO VERIFIED SOURCE YET", ground_truth="malware",
        url=None, local_name="ursnif.pcap", expected_bytes=None,
        confidence_note="NO DATA. Fill in this url manually if an Ursnif pcap with a verifiable origin is found.",
    ),
]

# --- EXTENDED_MANIFEST, added 25 August 2026 -- broader malware-family coverage & groundwork for
#     a coherent Layer 2 population (see the "insider threat, known population" discussion). ---
#
#     IMPORTANT: both are hosted on mcfp.felk.cvut.cz, like Dridex above -- if the network allowlist of
#     the environment blocks this domain (as happened in the development sandbox where this notebook was
#     written, see the [FAILED] message if it appears), the download MUST be run from your own machine.
#     The sizes/URLs below were verified through the directory listing, NOT through a full byte-exact
#     download (unlike the 5 PILOT_MANIFEST entries above) -- hence their confidence_note is MEDIUM, not HIGH.
EXTENDED_MANIFEST = [
    DatasetEntry(
        key="iot23_capture1", family="IoT-23 scenario 1 (malware family not yet read from the scenario README.md)",
        ground_truth="malware",
        url="https://mcfp.felk.cvut.cz/publicDatasets/IoT-23-Dataset/IndividualScenarios/CTU-IoT-Malware-Capture-1-1/2018-05-09-192.168.100.103.pcap",
        local_name="iot23_capture1.pcap", expected_bytes=None,
        confidence_note=(
            "MEDIUM CONFIDENCE — size (~139MB) from the directory listing, NOT yet verified "
            "byte-exact by a real download. The exact malware family name is in "
            "IndividualScenarios/CTU-IoT-Malware-Capture-1-1/README.md at the source -- READ that file "
            "before writing the family name in the paper; do not assume it from the scenario number alone."
        ),
        source_dataset="IoT-23 (Stratosphere/Aposemat)",
    ),
]

for e in PILOT_MANIFEST:
    print(f"[{e.key:12s}] {e.family:40s} -> {e.confidence_note}")
print()
print("-- EXTENDED_MANIFEST (25 August 2026, additional datasets) --")
for e in EXTENDED_MANIFEST:
    print(f"[{e.key:16s}] {e.family:55s} -> {e.confidence_note}")

# The combined manifest actually used by the download/Maltrail/Zeek/summary loops below.
# The original PILOT_MANIFEST is NOT changed at all -- this is purely added alongside it.
ALL_MANIFEST = PILOT_MANIFEST + EXTENDED_MANIFEST


In [ ]:
def download_entry(entry: DatasetEntry, dest_dir: Path, chunk_size: int = 1 << 16) -> Optional[Path]:
    '''Download one manifest entry with a byte-integrity check (if expected_bytes is available).'''
    if entry.url is None:
        print(f"[SKIP] {entry.key}: no URL (see confidence_note)")
        return None

    dest_path = dest_dir / entry.local_name
    if dest_path.exists():
        size = dest_path.stat().st_size
        if entry.expected_bytes is None or size == entry.expected_bytes:
            print(f"[ALREADY PRESENT] {entry.key}: {dest_path} ({size:,} bytes)")
            return dest_path
        else:
            print(f"[RE-DOWNLOAD] {entry.key}: local size {size:,} != expected {entry.expected_bytes:,}")

    print(f"[DOWNLOAD] {entry.key} <- {entry.url}")
    with requests.get(entry.url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(dest_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                f.write(chunk)

    actual = dest_path.stat().st_size
    if entry.expected_bytes is not None:
        status = "MATCH" if actual == entry.expected_bytes else "!! MISMATCH !!"
        print(f"         size: {actual:,} bytes (expected {entry.expected_bytes:,}) -> {status}")
    else:
        print(f"         size: {actual:,} bytes (no byte-exact expectation; rough plausibility check only)")

    return dest_path


downloaded = {}
for entry in ALL_MANIFEST:   # <- extended 25 August 2026, previously PILOT_MANIFEST only
    try:
        path = download_entry(entry, PCAP_RAW_DIR)
        if path:
            downloaded[entry.key] = path
    except Exception as exc:
        print(f"[FAILED] {entry.key}: {exc!r}")
        print("        (one failure does not stop the other entries — check this entry's connection/URL)")
        print("        If this is an EXTENDED_MANIFEST entry (mcfp.felk.cvut.cz) and the notebook runs")
        print("        in a cloud environment that blocks it: download it manually on your own machine,")
        print("        put the file in data/pcap_raw/ with the matching local_name, then re-run this")
        print("        cell -- the [ALREADY PRESENT] branch above will use it automatically.")

print()
print(f"Done: {len(downloaded)}/{len(ALL_MANIFEST)} pcaps (pilot + extended) available.")


### B.2 — Optional: Cridex & Shifu (`.7z` extraction needed)

Run this cell after the 5-file pilot pipeline above is shown to work. Needs `py7zr` (`pip install py7zr`) or a `7z`/`7zz` binary on PATH (`brew install sevenzip` or `brew install p7zip`).


In [ ]:
# Optional — run later, not part of the core pilot
RUN_OPTIONAL_EXTRACTION = False  # set to True when ready to scale

if RUN_OPTIONAL_EXTRACTION:
    for entry in OPTIONAL_MANIFEST:
        if entry.url is None:
            print(f"[SKIP] {entry.key}: {entry.confidence_note}")
            continue
        archive_path = download_entry(entry, PCAP_RAW_DIR)
        if archive_path and entry.needs_extraction:
            try:
                import py7zr
                with py7zr.SevenZipFile(archive_path, mode="r") as z:
                    z.extractall(path=PCAP_EXTRACTED_DIR)
                print(f"[EXTRACT] {entry.key} -> {PCAP_EXTRACTED_DIR}")
            except ImportError:
                print("py7zr is not installed. `pip install py7zr` and re-run this cell,")
                print("or extract manually: 7z x", archive_path)
else:
    print("RUN_OPTIONAL_EXTRACTION = False -> skipped (not yet time, see the note above).")


## Section C — Maltrail (rule-based detection)

The recipe in this section **has been verified empirically** (see the status table above) — the result of the guided discovery session in a terminal on the authors' Mac on 24 August 2026, no longer a guess.

### C.1 — Installation (if not present yet)

```bash
git clone https://github.com/stamparm/maltrail.git tools/maltrail
```

The sensor used is `tools/maltrail/old/sensor.py` (Python) — **not** the official Rust sensor (`tools/maltrail/sensor/`), because the latter fails to build on macOS arm64 (Linux-only `AF_PACKET` code not `cfg`-gated). Dependency needed by `old/sensor.py`: `pcapy-ng` (a native extension, shown to install cleanly from a prebuilt wheel on Apple Silicon, without compilation).

```bash
pip install pcapy-ng
```

For the public repository: check out upstream commit `7ea90920` and apply `tools/maltrail-patch/maltrail_patches.diff` (see `tools/maltrail-patch/README.md`); the patch cells below then report "already-patched".


In [ ]:
MALTRAIL_DIR = BASE_DIR / "tools" / "maltrail"
SENSOR_PY = MALTRAIL_DIR / "old" / "sensor.py"
MALTRAIL_LOCAL_CONF = MALTRAIL_DIR / "maltrail.local.conf"

if not MALTRAIL_DIR.exists():
    print(f"Maltrail is not present at {MALTRAIL_DIR}")
    print("Run `git clone https://github.com/stamparm/maltrail.git tools/maltrail` first.")
elif not SENSOR_PY.exists():
    print(f"{SENSOR_PY} not found.")
    print("The Maltrail repo structure may have changed again since this notebook was written (24 August 2026) —")
    print("re-check `ls tools/maltrail` and `ls tools/maltrail/old` before continuing; do not assume.")
else:
    print("old/sensor.py found:", SENSOR_PY)


### C.1b — Offline-replay timestamp patch (further drift investigation, 25 August 2026)

**Context**: the `PROCESS_COUNT 1` fix (C.2) was confirmed NOT ENOUGH -- the audit of section C.5 showed that the
byte-identical `zeus.pcap` still produced different alert counts (16/17/18) even with `PROCESS_COUNT` locked to 1
(so it is not only about multiprocessing). Further investigation traced DIRECTLY into `old/sensor.py`: the code
comment says `pcapy-ng returns unusable header timestamps on Python 3, so the wall clock is substituted` -- **this
claim was re-tested empirically** (a synthetic 5-packet pcap with microsecond-precision timestamps, read back with
`pcapy.open_offline()` + `header.getts()`) and **could NOT be verified** for `pcapy-ng` 2.1.0 on Python 3.11 (Linux)
-- `getts()` does return the CORRECT pcap timestamp, exactly down to the microsecond. Presumably the comment was
written for an older `pcapy-ng` version (the `pcapy-ng` library is maintained by the same author as Maltrail,
`stamparm` -- the bug was probably fixed in a newer release but the workaround was never removed from
`old/sensor.py`, which is already legacy).

**The two cells below**:
1. **Self-test** -- run EXACTLY the same test through this folder's Python `.venv` (not only in the development
   sandbox), so we KNOW (rather than assume) whether `getts()` is valid in the actual environment (Python 3.9.6 on
   the Mac; the `pcapy-ng` version may differ from the one tested in the sandbox).
2. **Apply the patch** -- replace the line `sec, usec = ... time.time() ...` in `old/sensor.py` so that, ONLY in
   offline mode (`--offline`, i.e. all our experiments), it uses `header.getts()` (the original pcap timestamp)
   instead of the wall clock, WITH an automatic fallback to the wall clock (+ a one-time warning) if `getts()` turns
   out to return implausible values in that environment -- so THIS PATCH IS SAFE even if the self-test says "not
   valid": on failure its behaviour is identical to before patching (it does not make things worse). Live capture
   (not `--offline`) is NOT touched at all. Idempotent -- safe to re-run many times; it never patches twice.

**Tested before delivery**: the patch was applied to an upstream `tools/maltrail` clone (confirmed byte-identical to
the one on the authors' Mac), `py_compile` passes, and 4 logic scenarios (valid timestamp, garbage timestamp, live
capture mode, future timestamp) all pass isolated tests. **NOT yet verified running in full inside the real
`sensor.py`** (the development sandbox did not have a complete Maltrail) -- which is why section C.5 below (now run
N times, not just 2x) MUST still be re-run after this as the final proof.

In [ ]:
# --- C.1b (1/2) -- Self-test: is header.getts() valid in this venv? ---
_GETTS_SELFTEST_SCRIPT = r'''
import struct, sys, time
from pathlib import Path

def build(path, n=5, base_ts=1300000000.0, step=500.123456):
    expected = []
    with open(path, "wb") as f:
        f.write(struct.pack("<IHHiIII", 0xa1b2c3d4, 2, 4, 0, 0, 65535, 1))  # global header, linktype=DLT_EN10MB
        payload = b"\\x00" * 14  # packet content is irrelevant -- only the header timestamp read is tested
        for i in range(n):
            ts = base_ts + i * step
            sec, usec = int(ts), int(round((ts - int(ts)) * 1e6))
            expected.append((sec, usec))
            f.write(struct.pack("<IIII", sec, usec, len(payload), len(payload)))
            f.write(payload)
    return expected

try:
    import pcapy
except Exception as exc:
    print("IMPORT_FAILED:%s" % exc)
    sys.exit(0)

p = Path(sys.argv[1])
expected = build(p)
cap = pcapy.open_offline(str(p))
got = []
while True:
    header, packet = cap.next()
    if header is None:
        break
    got.append(header.getts())
try:
    p.unlink()
except OSError:
    pass
print("EXPECTED:%s" % expected)
print("GOT:%s" % got)
print("MATCH:%s" % (got == expected))
'''

GETTS_WORKS_HERE = None
if MALTRAIL_DIR.exists() and SENSOR_PY.exists():
    _selftest_python = BASE_DIR / ".venv" / "bin" / "python3"
    if not _selftest_python.exists():
        print(f"[!] {_selftest_python} does not exist -> using sys.executable ({sys.executable}) for this self-test.")
        _selftest_python = Path(sys.executable)

    _selftest_pcap = MALTRAIL_OUT_DIR / "_getts_selftest.pcap"
    proc = subprocess.run([str(_selftest_python), "-c", _GETTS_SELFTEST_SCRIPT, str(_selftest_pcap)],
                           capture_output=True, text=True, timeout=30)
    print(proc.stdout.strip())
    if proc.returncode != 0 or proc.stderr.strip():
        print("stderr:", proc.stderr.strip()[:500])

    if "MATCH:True" in proc.stdout:
        GETTS_WORKS_HERE = True
        print()
        print("[OK] header.getts() RETURNS THE ORIGINAL TIMESTAMP CORRECTLY in this environment.")
        print("     -> The patch in the next cell (C.1b 2/2) will most likely be fully ACTIVE (not the fallback).")
    elif "MATCH:False" in proc.stdout:
        GETTS_WORKS_HERE = False
        print()
        print("[!] header.getts() does NOT match the expected values -- the 'unusable' claim in the sensor.py comment")
        print("    is PROVEN TRUE in this environment. The patch in the next cell will automatically FALL BACK to the")
        print("    wall clock (through its built-in sanity check) -- drift will most likely REMAIN,")
        print("    further investigation needed (report the EXPECTED/GOT output above).")
    else:
        print()
        print("[!] The self-test did not produce a clear MATCH: verdict (check IMPORT_FAILED/stderr above).")
else:
    print("MALTRAIL_DIR/SENSOR_PY incomplete -> self-test skipped. Check cell C.1 above.")


In [ ]:
# --- C.1b (2/2) -- Apply the patch: use header.getts() in offline mode, safe fallback to the wall clock ---
_SENSOR_PATCH_OLD_BLOCK = '''        try:
            # NOTE: pcapy-ng returns unusable header timestamps on Python 3, so the wall clock
            # is substituted. https://github.com/helpsystems/pcapy/issues/37#issuecomment-530795813
            sec, usec = [int(_) for _ in ("%.6f" % time.time()).split('.')]
'''

_SENSOR_PATCH_NEW_BLOCK = '''        try:
            # NOTE (original): pcapy-ng returns unusable header timestamps on Python 3, so the wall clock
            # is substituted. https://github.com/helpsystems/pcapy/issues/37#issuecomment-530795813
            # PATCH 25 August 2026 (investigation of offline-replay non-determinism, see README.md):
            # the claim above could NOT be verified empirically for pcapy-ng 2.1.0 (Python 3.11, Linux) --
            # header.getts() does return the original pcap timestamp correctly (tested directly with a
            # synthetic 5-packet pcap, exact match down to the microsecond -- see README.md and the C.1b
            # self-test cell). Used HERE only for OFFLINE mode (config.pcap_file set) so that replaying
            # the SAME pcap yields the SAME time values on every run -- a requirement for experiment
            # reproducibility. Live capture STILL uses the wall clock (original behaviour NOT changed) --
            # the "unusable" claim for live capture has not been re-verified; do not assume it is safe.
            # Automatic fallback to the wall clock if getts() returns implausible values (e.g. another
            # pcapy-ng version really is broken in that environment).
            sec = usec = None
            if config.pcap_file:
                try:
                    _ts_sec, _ts_usec = header.getts()
                    if 631152000 <= _ts_sec <= time.time() + 86400 and 0 <= _ts_usec <= 999999:
                        sec, usec = int(_ts_sec), int(_ts_usec)
                except Exception:
                    pass
            if sec is None:
                if config.pcap_file and not getattr(packet_handler, "_ts_fallback_warned", False):
                    packet_handler._ts_fallback_warned = True
                    print("[!] PATCH offline-timestamp: header.getts() is not valid in this environment -- "
                          "falling back to the wall clock (non-deterministic drift is STILL possible, see "
                          "README.md 'Investigation of the Zeus alert-count drift').")
                sec, usec = [int(_) for _ in ("%.6f" % time.time()).split('.')]
'''

_PATCH_MARKER = "PATCH offline-timestamp"


def patch_sensor_offline_timestamps() -> str:
    text = SENSOR_PY.read_text()
    if _PATCH_MARKER in text:
        return "already-patched"
    if _SENSOR_PATCH_OLD_BLOCK not in text:
        return "FAILED-original-block-not-found"
    SENSOR_PY.write_text(text.replace(_SENSOR_PATCH_OLD_BLOCK, _SENSOR_PATCH_NEW_BLOCK, 1))
    return "patched"


if MALTRAIL_DIR.exists() and SENSOR_PY.exists():
    _result = patch_sensor_offline_timestamps()
    print("offline-timestamp patch status:", _result)
    if _result == "FAILED-original-block-not-found":
        print("[!] The original lines being searched for were NOT FOUND -- old/sensor.py has probably changed")
        print("    from the version checked on 25 August 2026. Do NOT assume it is safe; investigate")
        print("    before continuing to C.2/C.3.")
    elif _result == "patched":
        print("The line `sec, usec = ... time.time() ...` in old/sensor.py (offline mode) has been replaced")
        print("by header.getts() with a safe fallback. Continue to C.2 (build config) as usual.")
    else:
        print("Already patched in an earlier run (idempotent, never patched twice).")
else:
    print("MALTRAIL_DIR/SENSOR_PY incomplete -> patch skipped. Check cell C.1 above.")


### C.1b (continued) — Relative-timestamp fallback fix (tinba.pcap non-determinism, 28 August 2026)

**Context**: after the fix of sections C.3/C.5 (log-file discovery — see the C.3 markdown), the C.5 run for
zeus.pcap is now genuinely deterministic (`[5697, 5697, 5697, 5697, 5697]`, 5 identical runs). But comparing the
26 vs 28 August runs for `tinba.pcap` (the `[diag-v3]` data in section C.3):

```
26 August: total=10827  dropped_throttle=9825  written=1002
28 August: total=10827  dropped_throttle=8823  written=2004
```

`total` (number of `log_event` calls) is **identical** — the trail matching itself is stable. But
`dropped_throttle`/`written` differ by almost 2x. **[HIGH CONFIDENCE]** this is a bug/non-determinism separate
from the ones already fixed.

**Root cause**: extracting the raw bytes of `tinba.pcap` directly shows per-packet `ts_sec` ranging from **0 to
~1755** (anchored near the 1970-01-01 epoch, not the real capture date). These values FAIL the sanity-check floor
of the Fix #2 offline-timestamp patch (`>= 631152000`, year 1990), so they automatically **fall back to the real
wall clock** (`time.time()`). The per-second throttle (`current_bucket = sec // config.PROCESS_COUNT`) then depends
on when/how fast the replay ACTUALLY RUNS on the machine — not on the pcap content — and so differs per run.

**The difference from zeus**: zeus.pcap has REAL timestamps (2012-05-29) that pass the sanity check, so it has been
deterministic since Fix #2. tinba.pcap has VALID timestamps but the **wrong anchor** (near-epoch, probably a
synthetic/relative pcap, not an original capture) — this was not yet handled.

**Fix**: those near-epoch timestamps are NOT garbage — they are read directly from the pcap file bytes, so they are
always EXACTLY the same every time the same file is read again (deterministic by construction). They only need to be
shifted into the range that passes the sanity check, **not replaced by the wall clock**. One new branch was added:
if `ts_sec` is valid (0 ≤ ts_sec < floor, usec within range) but below the floor, shift it by a **FIXED offset**
(the sanity-check floor itself, `631152000`) — not `time.time()`. This preserves two things at once: (1) it is
deterministic, because the offset is constant, not the wall clock; (2) the original INTER-PACKET gaps are preserved
(only the anchor shifts, not the structure) — the per-second throttle depends on these gaps, not on absolute values,
so the result is representative of the pcap itself, not an artefact of the offset. Real timestamps (zeus and others,
already passing the primary sanity check) are not touched at all by this new branch — confirmed by reading the real
`zeus.pcap` below (93,141 packets, all through the old branch).

Tested before delivery: (a) the patch — idempotency (patched → already-patched; note: the first attempt passed
"patched" but FAILED "already-patched" because the marker string turned out not to be actually embedded in the new
block's text — caught by this test itself and fixed before continuing), `py_compile` passes, a clean diff of 1
place, the Fix #2 prerequisite checked explicitly (`FAILED-fix2-prerequisite-missing` when tested without Fix #2),
the failure path (anchor deliberately broken) confirmed NOT to write anything; (b) 11 isolated logic scenarios —
real timestamps untouched, near-epoch timestamps shifted correctly (including `ts_sec=0` and `ts_sec=1755`, the
bounds of the real tinba pcap), 20x calls with identical input give identical output (determinism), the boundary
exactly at the floor (not shifted twice), 1 second below the floor, negative `ts_sec` still falls back (not wrongly
passed), broken `usec` still falls back, exceptions from `getts()` still fall back without crashing, live mode
untouched; (c) **direct confirmation with `pcapy` + the real `tinba.pcap`/`zeus.pcap`** (not just mocks) — 22,000
tinba packets read, raw ts_sec range exactly 0..1755 (matching the earlier manual extraction), the per-second
throttle buckets of two simulated "runs" with `now` (wall clock) ~11.5 days apart are **exactly identical** — direct
proof that this fix removes the root of the non-determinism; zeus.pcap (93,141 packets) confirmed to go 100% through
the old branch, not touched at all by this new patch.

**Not yet verified in a real run** — it needs to be run on the Mac (order unchanged: C.1b → **this fix** → C.2 →
C.1c → C.1d → C.1e → C.3 → C.4 → C.5), then compare the tinba `[diag-v3]` between runs (run C.3 twice in a row, or
look directly at C.5 with `DETERMINISM_CHECK_KEYS` including `"tinba"`) to confirm determinism in the real
environment, not only in the sandbox.


In [ ]:
"""
Offline-timestamp patch v2 (28 August 2026) -- targets old/sensor.py, COMPLEMENTING (not replacing) the
original offline-timestamp patch (Fix #2, section C.1b, marker 'PATCH offline-timestamp').

Context: the C.5 (determinism) runs of 26 & 28 August showed tinba.pcap is NOT deterministic -- the
total number of log_event calls is identical (10827=10827) but dropped_throttle/written differ widely
(9825/1002 vs 8823/2004). Root cause: tinba.pcap's original timestamps turn out to be ts_sec ~0..1755 (near
the 1970-01-01 epoch) -- they FAIL the Fix #2 sanity-check floor (>= 631152000, year 1990), so they fall back
to the REAL wall clock (`time.time()`), which differs per run depending on the real replay speed on the
machine. The per-second throttle (`sec // PROCESS_COUNT`) therefore becomes non-deterministic too.

The ts_sec 0..1755 timestamps are NOT garbage/exceptions -- they are valid, well-formed and (most
importantly) INTERNALLY CONSISTENT & deterministic: read directly from the pcap file bytes, so ALWAYS
identical on every read. Only their anchor is wrong (near 1970, not the real capture date, probably a
synthetic/relative pcap). Solution: if ts_sec is valid BUT below the sanity-check floor, shift it by a FIXED
OFFSET (not the wall clock) -- this stays deterministic (constant offset) AND preserves the original
inter-packet gaps of the pcap (which determine the per-second throttle result), unlike the wall-clock
fallback, which replaces the timing ENTIRELY with the replay process's time.

Prerequisite: the original offline-timestamp patch (Fix #2, marker 'PATCH offline-timestamp') MUST already
be applied to the same file -- the notebook always runs section C.1b before this cell, so the prerequisite
is always met in the normal flow.
"""
from pathlib import Path

_MARKER = "PATCH offline-timestamp-relative-fallback"
_V1_MARKER = "PATCH offline-timestamp"

_OLD_BLOCK = '''            sec = usec = None
            if config.pcap_file:
                try:
                    _ts_sec, _ts_usec = header.getts()
                    if 631152000 <= _ts_sec <= time.time() + 86400 and 0 <= _ts_usec <= 999999:
                        sec, usec = int(_ts_sec), int(_ts_usec)
                except Exception:
                    pass
            if sec is None:
'''

_NEW_BLOCK = '''            sec = usec = None
            if config.pcap_file:
                try:
                    _ts_sec, _ts_usec = header.getts()
                    if 631152000 <= _ts_sec <= time.time() + 86400 and 0 <= _ts_usec <= 999999:
                        sec, usec = int(_ts_sec), int(_ts_usec)
                    elif 0 <= _ts_sec < 631152000 and 0 <= _ts_usec <= 999999:
                        # PATCH offline-timestamp-relative-fallback (28 August 2026, investigation of
                        # tinba.pcap non-determinism, see README.md "Relative-timestamp fallback fix
                        # (tinba)"): a VALID timestamp
                        # (not an exception/garbage) but below the sanity-check floor -- found
                        # in tinba.pcap (ts_sec 0..~1755, anchored near the 1970-01-01 epoch, not the
                        # real capture date -- probably a synthetic/relative pcap). Such timestamps
                        # are still INTERNALLY CONSISTENT & deterministic (read straight from the file
                        # bytes, identical on every read) -- shifted by a FIXED OFFSET (the
                        # sanity-check floor itself, 631152000 = 1990-01-01) so they pass the
                        # sanity-check range WITHOUT the wall clock (which differs per run). The original
                        # inter-packet gaps are preserved -- the per-second throttle depends on those
                        # gaps, not on absolute values, so replaying the same pcap is deterministic again.
                        sec, usec = int(_ts_sec) + 631152000, int(_ts_usec)
                except Exception:
                    pass
            if sec is None:
'''

_ALL_PAIRS = ((_OLD_BLOCK, _NEW_BLOCK),)


def patch_sensor_relative_timestamp_fallback(sensor_py_path: Path) -> str:
    """
    Returns one of: "already-patched", "patched", "FAILED-fix2-prerequisite-missing",
    "FAILED-block-not-found".
    """
    text = sensor_py_path.read_text()
    if _MARKER in text:
        return "already-patched"
    if _V1_MARKER not in text:
        return "FAILED-fix2-prerequisite-missing"
    for old, new in _ALL_PAIRS:
        if old not in text:
            return "FAILED-block-not-found"
    for old, new in _ALL_PAIRS:
        text = text.replace(old, new, 1)
    sensor_py_path.write_text(text)
    return "patched"


if MALTRAIL_DIR.exists() and SENSOR_PY.exists():
    _result_rel_ts = patch_sensor_relative_timestamp_fallback(SENSOR_PY)
    print("offline-timestamp-relative-fallback patch status:", _result_rel_ts)
    if _result_rel_ts == "FAILED-fix2-prerequisite-missing":
        print("[!] The offline-timestamp patch (Fix #2, cell C.1b above) has not been applied -- run")
        print("    cell C.1b (2/2) before this cell.")
    elif _result_rel_ts == "FAILED-block-not-found":
        print("[!] The block being searched for was NOT FOUND -- old/sensor.py has probably changed from")
        print("    the version checked on 28 August 2026. Do NOT assume it is safe; investigate")
        print("    before continuing.")
    elif _result_rel_ts == "patched":
        print("The relative-timestamp fallback branch (for pcaps like tinba.pcap, near-epoch ts_sec)")
        print("has been added. Real timestamps (zeus and others) are untouched. Continue to C.2 as usual.")
    else:
        print("Already patched in an earlier run (idempotent, never patched twice).")
else:
    print("MALTRAIL_DIR/SENSOR_PY incomplete -> patch skipped. Check cell C.1 above.")


### C.1c — SYN burst-skip diagnostic instrumentation (further investigation of the alert collapse, 26 August 2026)

**Context**: the latest run (after Fix #2) shows zeus & iot23_capture1 collapsing to **0 alerts** (from a history
of 16-18 & 3), while tinba rose to 1741. The first guess -- `trails.csv` changing content between test sessions --
**was checked directly against `~/.maltrail/trails.csv` on the Mac and REFUTED**: that file (and all its
derivatives) has not changed since 24 August 2026 14:43 UTC -- EXACTLY the same snapshot was used in all of
yesterday's runs, both the 16-18-alert runs and those that collapsed to 0. That snapshot has been frozen as a
reference in the `trails_snapshots/` folder (see the `MANIFEST_*.txt` there).

Since it is not the blacklist data, the suspicion now shifts to **the code's own behaviour after Fix #2**. In
`old/sensor.py`, the blacklist-matching path for SYN packets goes through a *burst-skip* logic -- `_last_syn` --
which compares the tuple `(sec, src_ip, src_port, dst_ip, dst_port)` of the current packet with the previous SYN
packet; if identical, the packet is `return`ed BEFORE it is ever matched against the blacklist. `sec` there now
holds the ORIGINAL pcap timestamp (Fix #2), no longer the wall clock -- and the original timestamps of old
botnet scan/C2 traffic can be much denser/more repetitive than the processing wall clock. **This is only a
suspicion from reading the code, NOT yet tested empirically** -- this cell adds pure counters that do NOT change
detection behaviour at all, so the suspicion can be proven or refuted with real numbers instead of guesses.

**How to read the results**: whenever `sensor.py` finishes processing a pcap in `--offline` mode, a line
`[diag] SYN counters for '<pcap>': total=... burst_skipped=... trail_hit=... logged_skipped=...
alert_emitted=...` appears in section C.3 (and C.5, since both call `sensor.py`). Meaning:
- `total` = all SYN packets examined.
- `burst_skipped` = how many were skipped BEFORE being matched against the blacklist (a candidate cause of the collapse).
- `trail_hit` = how many matched the blacklist (whether or not they ended up as an alert).
- `logged_skipped` = those matching the blacklist BUT skipped by the second dedup (`_last_logged_syn`).
- `alert_emitted` = those that actually became an alert line -- this should add up to `n_lines` above.

If `burst_skipped` for zeus/iot23 is (proportionally) much larger than for tinba, that is strong evidence the
suspicion above is right. If `trail_hit` is already 0 from the start (rather than a high `burst_skipped`), the
problem lies elsewhere (e.g. the blacklist matching itself), not in burst-skip -- report all the numbers back; do
not assume either explanation is automatically right.

In [ ]:
"""
Diagnostic patch (temporary, offline-only): counts how many SYN packets pass/are skipped
at each stage of the burst-dedup -> trail-match -> logged-dedup -> alert-emit pipeline in old/sensor.py.
Tested in isolation before being added to the notebook. Independent of the Fix #2 patch
(patch_sensor_offline_timestamps) -- it touches different lines (552-608, 1636-1650 vs 1494/1554),
so it is safe to apply on top of Fix #2 without conflicts.
"""
from pathlib import Path

_MARKER = "PATCH diagnostic-syn-counters"

_OLD_BLOCK_1 = "_last_syn = None\n"

_NEW_BLOCK_1 = (
    "_last_syn = None\n"
    "\n"
    "# DIAGNOSTIC (temporary, offline-only) -- counts how many SYN packets reach each stage of the\n"
    "# blacklist-match / burst-dedup pipeline, to investigate the alert-count collapse observed after\n"
    "# the header.getts() offline-timestamp patch (PATCH offline-timestamp). Pure counting, changes no\n"
    "# detection behaviour. Marker: '" + _MARKER + "'.\n"
    "_DIAG_SYN_COUNTERS = {\"total\": 0, \"burst_skipped\": 0, \"trail_hit\": 0, \"logged_skipped\": 0, \"alert_emitted\": 0}\n"
)

_OLD_BLOCK_2 = '''            if flags == 2:  # SYN set (only)
                _ = _last_syn
                _last_syn = (sec, src_ip, src_port, dst_ip, dst_port)
                if _ == _last_syn:  # skip bursts
                    return

                if dst_ip in trails or addr_port(dst_ip, dst_port) in trails:
                    _ = _last_logged_syn
                    _last_logged_syn = _last_syn
                    if _ != _last_logged_syn:
                        trail = addr_port(dst_ip, dst_port)
                        if trail not in trails:
                            trail = dst_ip
                        if not any(_ in trails[trail][0] for _ in ("attacker",)) and not ("parking site" in trails[trail][0] and dst_port not in (80, 443)):
                            # IPORT iff the matched key is the addr_port form (not the bare IP). The old
                            # `':' not in trail` check mis-typed EVERY IPv6 IP-trail as IPORT (v6 addrs contain ':').
                            log_event((sec, usec, src_ip, src_port, dst_ip, dst_port, PROTO.TCP, TRAIL.IPORT if trail != dst_ip else TRAIL.IP, trail, trails[trail][0], trails[trail][1]), packet)

                elif (src_ip in trails or addr_port(src_ip, src_port) in trails) and dst_ip != localhost_ip:
                    _ = _last_logged_syn
                    _last_logged_syn = _last_syn
                    if _ != _last_logged_syn:
                        trail = addr_port(src_ip, src_port)
                        if trail not in trails:
                            trail = src_ip
                        if not any(_ in trails[trail][0] for _ in ("malware",)):
                            log_event((sec, usec, src_ip, src_port, dst_ip, dst_port, PROTO.TCP, TRAIL.IPORT if trail != src_ip else TRAIL.IP, trail, trails[trail][0], trails[trail][1]), packet)
'''

_NEW_BLOCK_2 = '''            if flags == 2:  # SYN set (only)
                _ = _last_syn
                _last_syn = (sec, src_ip, src_port, dst_ip, dst_port)
                if config.pcap_file:
                    _DIAG_SYN_COUNTERS["total"] += 1
                if _ == _last_syn:  # skip bursts
                    if config.pcap_file:
                        _DIAG_SYN_COUNTERS["burst_skipped"] += 1
                    return

                if dst_ip in trails or addr_port(dst_ip, dst_port) in trails:
                    if config.pcap_file:
                        _DIAG_SYN_COUNTERS["trail_hit"] += 1
                    _ = _last_logged_syn
                    _last_logged_syn = _last_syn
                    if _ != _last_logged_syn:
                        trail = addr_port(dst_ip, dst_port)
                        if trail not in trails:
                            trail = dst_ip
                        if not any(_ in trails[trail][0] for _ in ("attacker",)) and not ("parking site" in trails[trail][0] and dst_port not in (80, 443)):
                            # IPORT iff the matched key is the addr_port form (not the bare IP). The old
                            # `':' not in trail` check mis-typed EVERY IPv6 IP-trail as IPORT (v6 addrs contain ':').
                            if config.pcap_file:
                                _DIAG_SYN_COUNTERS["alert_emitted"] += 1
                            log_event((sec, usec, src_ip, src_port, dst_ip, dst_port, PROTO.TCP, TRAIL.IPORT if trail != dst_ip else TRAIL.IP, trail, trails[trail][0], trails[trail][1]), packet)
                    elif config.pcap_file:
                        _DIAG_SYN_COUNTERS["logged_skipped"] += 1

                elif (src_ip in trails or addr_port(src_ip, src_port) in trails) and dst_ip != localhost_ip:
                    if config.pcap_file:
                        _DIAG_SYN_COUNTERS["trail_hit"] += 1
                    _ = _last_logged_syn
                    _last_logged_syn = _last_syn
                    if _ != _last_logged_syn:
                        trail = addr_port(src_ip, src_port)
                        if trail not in trails:
                            trail = src_ip
                        if not any(_ in trails[trail][0] for _ in ("malware",)):
                            if config.pcap_file:
                                _DIAG_SYN_COUNTERS["alert_emitted"] += 1
                            log_event((sec, usec, src_ip, src_port, dst_ip, dst_port, PROTO.TCP, TRAIL.IPORT if trail != src_ip else TRAIL.IP, trail, trails[trail][0], trails[trail][1]), packet)
                    elif config.pcap_file:
                        _DIAG_SYN_COUNTERS["logged_skipped"] += 1
'''

_OLD_BLOCK_3 = '''                    elif config.pcap_file:
                        with _done_lock:
                            _done_count += 1
                        break
'''

_NEW_BLOCK_3 = '''                    elif config.pcap_file:
                        with _done_lock:
                            _done_count += 1
                        c = _DIAG_SYN_COUNTERS
                        print("[diag] SYN counters for '%s': total=%d burst_skipped=%d trail_hit=%d "
                              "logged_skipped=%d alert_emitted=%d" % (config.pcap_file, c["total"],
                              c["burst_skipped"], c["trail_hit"], c["logged_skipped"], c["alert_emitted"]))
                        break
'''


def patch_sensor_diagnostic_counters(sensor_py_path: Path) -> str:
    """
    Returns one of: "already-patched", "patched", "FAILED-block-not-found".
    """
    text = sensor_py_path.read_text()
    if _MARKER in text:
        return "already-patched"
    for old, new in ((_OLD_BLOCK_1, _NEW_BLOCK_1), (_OLD_BLOCK_2, _NEW_BLOCK_2), (_OLD_BLOCK_3, _NEW_BLOCK_3)):
        if old not in text:
            return "FAILED-block-not-found"
    text = text.replace(_OLD_BLOCK_1, _NEW_BLOCK_1, 1)
    text = text.replace(_OLD_BLOCK_2, _NEW_BLOCK_2, 1)
    text = text.replace(_OLD_BLOCK_3, _NEW_BLOCK_3, 1)
    sensor_py_path.write_text(text)
    return "patched"

print(patch_sensor_diagnostic_counters(SENSOR_PY))


### C.1d — Diagnostic instrumentation v2: trail-membership test BEFORE burst-skip + log_event breakdown per TRAIL type (26 August 2026)

**Context**: the `[diag]` results of C.1c (already run) give an important number for `zeus.pcap`:

```
total=32011  burst_skipped=18140  trail_hit=0  logged_skipped=0  alert_emitted=0
```

`trail_hit=0` **refutes** the simple form of the burst-skip hypothesis proposed earlier — if burst-skip were masking
real matches, the expected pattern is `trail_hit > 0` but masked in `logged_skipped`/`alert_emitted`. What happened
instead: of the 13,871 Zeus SYN packets that **passed** burst-skip and were actually checked against the
blacklist, **not a single one** matched — so the problem is not "a masked match" but "the match itself never
happens" in this code block. For IoT-23, by contrast, the numbers are consistent with history (5 alerts,
trail_hit=5) — so the instrumentation itself is correct, not a bug in the patch.

This v2 instrumentation addresses the two gaps left by v1:

1. **"Would the burst-skipped packets ACTUALLY match the blacklist if they were not skipped?"** — tested DIRECTLY
   (not assumed): EXACTLY the same trail-matching predicate as the original code (bare IP `dst_ip in trails` OR the
   combined form `addr_port(dst_ip, dst_port)`, i.e. `"IP:port"`) is evaluated BEFORE the line
   `if _ == _last_syn: return`. New counters: `would_match_total`, `would_match_dst`, `would_match_src`,
   `would_match_via_addr_port` (matches that ONLY arise through the IP+port combination, not through the bare IP
   alone), and `burst_skipped_would_have_matched`.
2. **"Did the old Zeus alerts come from paths OTHER than this TCP-SYN block?"** — `old/sensor.py` turns out to have
   MANY other places that check `trails` (the port-scan heuristic, the HTTP checks — sinkhole response/suspicious
   user agent/URL, the UDP/DNS path), all independent of the SYN block and all using the `sec`/`usec` changed by
   Fix #2. Our own earlier note on Zeus says `ALERT — "blackshades (malware)" + several heuristics` — that word
   "heuristic" most likely comes from these other paths, not from the SYN block instrumented in C.1c. To check this
   without instrumenting every block one by one, `log_event()` (the one function ALL the paths above go through)
   is wrapped to count how many times each `TRAIL.*` type (`IP`/`IPORT`/`DNS`/`HTTP`/`URL`/`UA`/`PORT`) is actually
   CALLED per pcap in offline mode. **Important note**: a recorded call does NOT guarantee that a line is actually
   written to `<pcap>.log` — `log_event()` itself has internal whitelist/`IGNORE_EVENTS`/per-second dedup gates
   (`current_bucket = sec // PROCESS_COUNT`) that this patch does not touch, so there may still be a gap between
   "called" and "written".

**On the suggestion (features from the 14 IPv4 header fields, aggregated src+port combinations)** — checked before
adding it: the predicate above ALREADY covers exactly the src/dst IP+port combination (`addr_port`), because that is
the real key form used in `trails.csv` (`would_match_via_addr_port` answers this explicitly). But the ~10 other IPv4
header fields besides IP/port (TTL, ToS, packet ID, fragmentation flags, fragment offset, checksum, IHL, total
length) are **not used anywhere** by Maltrail's existing blacklist-matching or heuristic logic — so adding them to
THIS diagnostic would not help explain why matches disappear. It is a valid idea for another direction
(header/packet-based Layer 2 features, in line with the "3 granularities" idea of an older doctoral draft), but not
for this Layer 1 bug — deliberately left out of this patch so its scope stays focused and testable. [a design
decision; corrections welcome if another gap is visible]

**How to read the results**: if `would_match_total` for zeus stays 0 (like v1's `trail_hit`) — no Zeus SYN packet
matches the blacklist through any path, at any point in the processing (so it is not about burst-skip at all) — the
root cause is elsewhere (e.g. non-SYN paths, or matching that used to happen via HTTP/UDP). If
`would_match_total > 0` while `trail_hit` (v1) stays 0 — burst-skip is PROVEN to swallow matches that should exist,
and `burst_skipped_would_have_matched` shows exactly how many. The `[diag-v2] log_event calls` line shows which
paths (`IP`/`IPORT`/`HTTP`/`DNS`/etc.) are actually called per pcap — if `IP`/`IPORT` are zero but other types are
non-zero for zeus, that is strong evidence the old alerts came from paths outside the SYN block suspected so far.

Tested before delivery: the `would_match_*` predicate was verified through 7 isolated logic scenarios (bare-IP
match, addr_port-only match, no match, localhost exclusion, dst+src both matching — all as expected); the
`log_event` wrapper was tested for offline-mode counting per type, no counting at all in live mode (zero
overhead/zero risk), and short/broken tuples not causing crashes. Applied on top of a `tools/maltrail` clone
already patched with Fix #2 + C.1c (v1) — confirmed not to conflict (the diff touches only 4 places: the counter
dict, the SYN block, the `import log_event` line, the EOF print line), `py_compile` passes, idempotency verified.
**Not yet verified in a real run** — it needs to be run on the Mac.


In [ ]:
"""
Diagnostic patch v2 (temporary, offline-only), continuing patch_sensor_diagnostic_counters (v1).
v1 already showed trail_hit=0 for zeus.pcap after Fix #2 -- but it only counts AFTER the
burst skip. This patch addresses the two gaps v1 leaves:

  1. "Would the burst-skipped packets ACTUALLY match the blacklist if they were not skipped?"
     -- tested DIRECTLY (not assumed) by repeating EXACTLY the same trail-matching predicate used by the
     original code (bare IP AND the addr_port "IP:port" form), counted BEFORE the line
     `if _ == _last_syn: return`. This also answers the question about src+port combinations:
     `_diag_would_match_via_addr_port` specifically marks matches that ONLY arise through the
     IP:port (addr_port) combination, not through the bare IP.
  2. "Did the old Zeus alerts come from paths OTHER than this TCP-SYN block?" -- log_event() (the one point
     that ALL paths go through: SYN, scan heuristics, HTTP, UDP/DNS, ICMP) is wrapped to count
     how many times each TRAIL.* type is actually called per pcap in offline mode.

Prerequisite: patch_sensor_diagnostic_counters (v1, marker 'PATCH diagnostic-syn-counters') MUST
already be applied to the same file -- the notebook always runs section C.1c before
this cell, so the prerequisite is always met in the normal flow. Independent of Fix #2
(patch_sensor_offline_timestamps) -- it does not touch the same lines.
"""
from pathlib import Path

_MARKER = "PATCH diagnostic-syn-counters-v2"
_V1_MARKER = "PATCH diagnostic-syn-counters"

# --- 1) extend the v1 counter dict -------------------------------------------------------------

_OLD_COUNTERS = (
    '_DIAG_SYN_COUNTERS = {"total": 0, "burst_skipped": 0, "trail_hit": 0, "logged_skipped": 0, "alert_emitted": 0}\n'
)

_NEW_COUNTERS = (
    '_DIAG_SYN_COUNTERS = {"total": 0, "burst_skipped": 0, "trail_hit": 0, "logged_skipped": 0, "alert_emitted": 0,\n'
    '                      "would_match_total": 0, "would_match_dst": 0, "would_match_src": 0,\n'
    '                      "would_match_via_addr_port": 0, "burst_skipped_would_have_matched": 0}\n'
    '# DIAGNOSTIC v2 (temporary, offline-only). Marker: \'' + _MARKER + '\'.\n'
)

# --- 2) SYN block: insert the trail-membership test BEFORE the burst-skip return ---------------

_OLD_SYN = '''            if flags == 2:  # SYN set (only)
                _ = _last_syn
                _last_syn = (sec, src_ip, src_port, dst_ip, dst_port)
                if config.pcap_file:
                    _DIAG_SYN_COUNTERS["total"] += 1
                if _ == _last_syn:  # skip bursts
                    if config.pcap_file:
                        _DIAG_SYN_COUNTERS["burst_skipped"] += 1
                    return
'''

_NEW_SYN = '''            if flags == 2:  # SYN set (only)
                _ = _last_syn
                _last_syn = (sec, src_ip, src_port, dst_ip, dst_port)
                _diag_would_match_dst = _diag_would_match_src = _diag_would_match_combo = False
                if config.pcap_file:
                    _DIAG_SYN_COUNTERS["total"] += 1
                    # DIAGNOSTIC v2: predicate IDENTICAL to the original check below (bare IP OR the
                    # addr_port "IP:port" form), evaluated here -- BEFORE the burst skip -- only to
                    # learn whether a packet about to be skipped would actually match the blacklist.
                    _diag_would_match_dst = dst_ip in trails or addr_port(dst_ip, dst_port) in trails
                    _diag_would_match_src = (src_ip in trails or addr_port(src_ip, src_port) in trails) and dst_ip != localhost_ip
                    _diag_would_match_combo = addr_port(dst_ip, dst_port) in trails or (addr_port(src_ip, src_port) in trails and dst_ip != localhost_ip)
                    if _diag_would_match_dst or _diag_would_match_src:
                        _DIAG_SYN_COUNTERS["would_match_total"] += 1
                        if _diag_would_match_dst:
                            _DIAG_SYN_COUNTERS["would_match_dst"] += 1
                        if _diag_would_match_src:
                            _DIAG_SYN_COUNTERS["would_match_src"] += 1
                        if _diag_would_match_combo:
                            _DIAG_SYN_COUNTERS["would_match_via_addr_port"] += 1
                if _ == _last_syn:  # skip bursts
                    if config.pcap_file:
                        _DIAG_SYN_COUNTERS["burst_skipped"] += 1
                        if _diag_would_match_dst or _diag_would_match_src:
                            _DIAG_SYN_COUNTERS["burst_skipped_would_have_matched"] += 1
                    return
'''

# --- 3) wrap log_event so that all paths (not just SYN) are counted ----------------------------

_OLD_LOGIMPORT = "from core.log import log_event\n"

_NEW_LOGIMPORT = (
    "from core.log import log_event\n"
    "\n"
    "# DIAGNOSTIC v2 (temporary, offline-only) -- log_event() is the ONE point that ALL trail-matching\n"
    "# paths in _process_packet go through (SYN, scan heuristics, HTTP, UDP/DNS, ICMP), not just the\n"
    "# TCP-SYN block instrumented in v1. Wrapping it here counts CALLS to log_event per TRAIL.* type --\n"
    "# a recorded call does NOT guarantee that a line is actually written to <pcap>.log (log_event itself\n"
    "# has internal whitelist/IGNORE_EVENTS/per-second dedup gates that this patch does not touch).\n"
    "# Marker: '" + _MARKER + "'.\n"
    "_DIAG_LOG_EVENT_CALLS = {}   # trail_type (str) -> number of calls, offline mode only\n"
    "_diag_orig_log_event = log_event\n"
    "def log_event(event_tuple, *args, **kwargs):\n"
    "    if config.pcap_file:\n"
    "        try:\n"
    "            _tt = str(event_tuple[7])\n"
    "            _DIAG_LOG_EVENT_CALLS[_tt] = _DIAG_LOG_EVENT_CALLS.get(_tt, 0) + 1\n"
    "        except Exception:\n"
    "            pass\n"
    "    return _diag_orig_log_event(event_tuple, *args, **kwargs)\n"
)

# --- 4) EOF summary print: extend with the new counters + the log_event breakdown --------------

_OLD_PRINT = '''                    elif config.pcap_file:
                        with _done_lock:
                            _done_count += 1
                        c = _DIAG_SYN_COUNTERS
                        print("[diag] SYN counters for '%s': total=%d burst_skipped=%d trail_hit=%d "
                              "logged_skipped=%d alert_emitted=%d" % (config.pcap_file, c["total"],
                              c["burst_skipped"], c["trail_hit"], c["logged_skipped"], c["alert_emitted"]))
                        break
'''

_NEW_PRINT = '''                    elif config.pcap_file:
                        with _done_lock:
                            _done_count += 1
                        c = _DIAG_SYN_COUNTERS
                        print("[diag] SYN counters for '%s': total=%d burst_skipped=%d trail_hit=%d "
                              "logged_skipped=%d alert_emitted=%d" % (config.pcap_file, c["total"],
                              c["burst_skipped"], c["trail_hit"], c["logged_skipped"], c["alert_emitted"]))
                        print("[diag-v2] would-match (pre-burst-skip) for '%s': would_match_total=%d "
                              "would_match_dst=%d would_match_src=%d would_match_via_addr_port=%d "
                              "burst_skipped_would_have_matched=%d" % (config.pcap_file,
                              c["would_match_total"], c["would_match_dst"], c["would_match_src"],
                              c["would_match_via_addr_port"], c["burst_skipped_would_have_matched"]))
                        if _DIAG_LOG_EVENT_CALLS:
                            print("[diag-v2] log_event calls for '%s': %s" % (config.pcap_file,
                                  ", ".join("%s=%d" % (k, v) for k, v in sorted(_DIAG_LOG_EVENT_CALLS.items()))))
                        else:
                            print("[diag-v2] log_event calls for '%s': (no calls at all)" % config.pcap_file)
                        break
'''


def patch_sensor_diagnostic_v2(sensor_py_path: Path) -> str:
    """
    Returns one of: "already-patched", "patched", "FAILED-v1-prerequisite-missing",
    "FAILED-block-not-found".
    """
    text = sensor_py_path.read_text()
    if _MARKER in text:
        return "already-patched"
    if _V1_MARKER not in text:
        return "FAILED-v1-prerequisite-missing"
    for old, new in ((_OLD_COUNTERS, _NEW_COUNTERS), (_OLD_SYN, _NEW_SYN),
                      (_OLD_LOGIMPORT, _NEW_LOGIMPORT), (_OLD_PRINT, _NEW_PRINT)):
        if old not in text:
            return "FAILED-block-not-found"
    text = text.replace(_OLD_COUNTERS, _NEW_COUNTERS, 1)
    text = text.replace(_OLD_SYN, _NEW_SYN, 1)
    text = text.replace(_OLD_LOGIMPORT, _NEW_LOGIMPORT, 1)
    text = text.replace(_OLD_PRINT, _NEW_PRINT, 1)
    sensor_py_path.write_text(text)
    return "patched"

print(patch_sensor_diagnostic_v2(SENSOR_PY))


### C.1e — Diagnostic instrumentation v3: getting through the 4 internal gates of `log_event()` in `core/log.py` (26 August 2026)

**Context**: the `[diag-v2]` results of C.1d (already run) give an important number for `zeus.pcap`:

```
[diag-v2] log_event calls for 'zeus.pcap': DNS=5554, UA=6, URL=258
```

This means `log_event()` is **called** 5,818 times for zeus — not zero, so the "no match at all" hypothesis (from
C.1c/v1) is also fully refuted at this point. But `zeus.log` still has 0 lines. Conclusion: the bottleneck has
definitely moved **inside** `log_event()` itself (`core/log.py`), no longer in the trail matching in `sensor.py`.
**[HIGH CONFIDENCE]** — this comes directly from the `[diag-v2]` numbers of an actual run, not from a guess.

`core/log.py` (461 lines) was read in full to map its internal path. There are 4 gates that can drop an event
BEFORE it gets written, in this order:

1. **`ignore_event()`** (from `core/ignore.py`) — purely config-based (`IGNORE_EVENTS_REGEX`, `IGNORE_EVENTS`),
   does not depend on `sec`/time at all.
2. **The whitelist gate** — `check_whitelisted(src_ip)` or `check_whitelisted(dst_ip)`, EXCEPT for events of type
   `TRAIL.DNS` (explicitly excluded in the original code — its own comment says "DNS requests/responses can't be
   whitelisted based on src_ip/dst_ip"). This is directly relevant because 5,554 of zeus's 5,818 calls are of
   type `DNS`.
3. **The "condense" queue** — if `info` contains one of the keywords
   `CONDENSE_ON_INFO_KEYWORDS = ("attacker", "reputation", "scanner", "user agent", "tor exit",
   "port scanning", "potential infection")`, the event goes into a per-`(src_ip, trail)` queue and is NOT yet
   written — it is written later by a background thread (`flush_condensed_events`, interval
   `CONDENSED_EVENTS_FLUSH_PERIOD`). This **matches exactly** the label `UA=6` ("user agent (suspicious)"
   contains the words "user agent").
4. **Per-second throttle** — `current_bucket = sec // config.PROCESS_COUNT`; within the same bucket, repeated
   `(ip, trail)` pairs are dropped. **This depends DIRECTLY on `sec`** — the value Fix #2 changed from the wall
   clock to the original pcap timestamp.

The v3 instrumentation counts exactly at which point each call is dropped — the counters:
`total`, `dropped_ignore_event`, `dropped_whitelisted`, `dropped_skip_write`, `queued_condense`,
`dropped_throttle`, `written`. Two parts, two different files (**the run order MUST be as follows**, see the
prerequisite note below):

- **Part 1/2** — `patch_log_diagnostic_gates()` targets `core/log.py` directly (a file NOT touched by any earlier
  patch in this experiment).
- **Part 2/2** — `patch_sensor_diagnostic_v3()` targets `old/sensor.py`, adding
  `from core import log as _diag_log_module` and then printing a `[diag-v3]` line at the same EOF point as
  `[diag]`/`[diag-v2]`.

**Unexpected finding from the testing process (not a bug in the patch, but an ORIGINAL property of Maltrail that
was not apparent before testing)**: the per-second throttle gate turns out to be **looser** than might be assumed.
The original code resets the dedup set `_thread_data.log_trails` to empty when entering a new second-bucket, BUT
does not immediately add the FIRST event of that bucket to the set — only the SECOND and later events are
checked/added. Consequence: the 1st and 2nd events with an identical `(ip, trail)` in the SAME second **always
pass**; only the 3rd (identical, same second) actually hits `dropped_throttle`. This was found through an isolated
scenario test (see below), not by reading the code alone — the initial assumption (2 occurrences are enough to
throttle) turned out to be wrong, and the test was corrected only after it failed the first time.
**[HIGH CONFIDENCE]** for this behaviour — verified by directly executing the original function, not by code
reading alone. This matters for interpreting `dropped_throttle` later: that number ONLY rises if there are >=3
events with exactly the same `(ip, trail)` within the same second.

**How to read the results**: if `dropped_whitelisted` is large for zeus — a strong candidate is that zeus's src/dst
IPs happen to be on the local whitelist (check `check_whitelisted`/the whitelist in the config). If
`queued_condense` is large and matches `UA=6` (not most of `DNS=5554`) — most zeus events do NOT go through the
condense queue, so that is not the main explanation for the majority of cases. If `dropped_ignore_event` is large
— check `IGNORE_EVENTS`/`IGNORE_EVENTS_REGEX` in your `maltrail.local.conf` (C.2); an unintended ignore rule may
cover zeus's IP range. If `written` is already > 0 but `zeus.log` is still empty — the problem has moved again,
this time to the file writing (`get_event_log_handle`, permissions, path) — not yet checked by this patch.

**Prerequisites & ordering risk (read before running)**: patch part 1/2 (`core/log.py`) MUST be applied before part
2/2 (`old/sensor.py`) is imported/run (i.e. before C.3/C.5) — the v3-patched `sensor.py` does
`from core import log as _diag_log_module` and reads `_diag_log_module._DIAG_LOG_GATE_COUNTERS` at runtime. If the
order is reversed, sensor.py fails with a CLEAR `AttributeError` (not a silent miscount) — deliberately designed
that way (failing fast is safer than being silently wrong for instrumentation whose data feeds research
conclusions). The code cell below already runs BOTH in the correct order, so the order does not need to be handled
manually — just run this cell ONCE before C.3, like C.1c/C.1d.

Tested before delivery: (a) `patch_log_diagnostic_gates` — idempotency (patched → already-patched), `py_compile`
passes, the diff is clean at exactly 5 points, the failure path (`FAILED-block-not-found`) was tested by
deliberately breaking one anchor and confirmed to write NOTHING to the file; (b) 8 isolated behaviour scenarios
importing the patched `core/log.py` and calling `log_event()` directly with mocked dependencies (`ignore_event`,
`check_whitelisted`, `CONDENSE_ON_INFO_KEYWORDS`) — covering each gate one by one, the DNS exception to the
whitelist, "live" mode (empty `config.pcap_file`) confirmed to have ZERO overhead/zero counting, and short tuples
deliberately tested NOT to change the original exception behaviour (`ValueError` still propagates, not silently
caught); (c) `patch_sensor_diagnostic_v3` — idempotency, the v2 prerequisite is checked
(`FAILED-v2-prerequisite-missing` when tested without v2), applied on top of a Fix#2+v1+v2 clone, the diff is clean
at 2 points, `py_compile` passes for all four patches combined (Fix#2+v1+v2+v3); (d) **a real bug was found and
fixed before this cell was finalised**: both patchers (log & sensor) define the module-level names `_MARKER` and
`_ALL_PAIRS` — if combined as-is into ONE notebook cell (one shared execution namespace), the second definition
would OVERWRITE the first, making `patch_log_diagnostic_gates()` read the sensor patch's marker/block pairs when
called. The sensor patch's variables were renamed (`_MARKER` → `_MARKER_V3_SENSOR`, `_ALL_PAIRS` →
`_ALL_PAIRS_V3_SENSOR`) specifically for the version combined in this cell, and the combined execution was
re-tested via `exec()` simulating exactly the cell contents (twice in a row, confirming `patched`/`patched` and
then `already-patched`/`already-patched`) before considering it safe. **Not yet verified in a real run** on a real
pcap — it needs to be run on the Mac, and the exported results reviewed afterwards.


In [ ]:
"""
Diagnostic patch v3, part 1/2 (temporary, offline-only) -- targets core/log.py, not sensor.py.

v2 proved that log_event() is CALLED thousands of times for zeus.pcap (DNS=5554, UA=6, URL=258) yet
zeus.log still has 0 lines -- so the bottleneck is INSIDE log_event() itself, no longer in the
trail matching in sensor.py. This function has gates that can drop an event BEFORE it is
written: ignore_event() (IGNORE_EVENTS/regex rules), the whitelist gate, the "condense" queue
(CONDENSE_ON_INFO_KEYWORDS -- an exact match for the label "user agent (suspicious)" of UA=6), and
the per-second dedup throttle (current_bucket = sec // PROCESS_COUNT, DIRECTLY dependent on the sec
changed by Fix #2). This patch counts exactly at which point each call is dropped.

Independent of all earlier patches (Fix #2, C.1c v1, C.1d v2) -- it touches a DIFFERENT file
(core/log.py, not old/sensor.py) entirely.
"""
from pathlib import Path

_MARKER = "PATCH diagnostic-log-gates"

_OLD_GLOBALS = '''_condensed_events = {}
_condensing_thread = None
_condensing_lock = threading.Lock()
_single_messages = set()
_thread_data = threading.local()
'''

_NEW_GLOBALS = '''_condensed_events = {}
_condensing_thread = None
_condensing_lock = threading.Lock()
_single_messages = set()
_thread_data = threading.local()

# DIAGNOSTIC v3 (temporary, offline-only) -- pinpoints exactly which internal gate of log_event()
# drops an event before it reaches the actual log write. Pure counting, no behaviour change.
# Marker: \'''' + _MARKER + '''\'.
_DIAG_LOG_GATE_COUNTERS = {"total": 0, "dropped_ignore_event": 0, "dropped_whitelisted": 0,
                           "dropped_skip_write": 0, "queued_condense": 0, "dropped_throttle": 0,
                           "written": 0}
'''

_OLD_IGNORE = '''        sec, usec, src_ip, src_port, dst_ip, dst_port, proto, trail_type, trail, info, reference = event_tuple
        if ignore_event(event_tuple):
            return
'''

_NEW_IGNORE = '''        sec, usec, src_ip, src_port, dst_ip, dst_port, proto, trail_type, trail, info, reference = event_tuple
        if config.pcap_file:
            _DIAG_LOG_GATE_COUNTERS["total"] += 1
        if ignore_event(event_tuple):
            if config.pcap_file:
                _DIAG_LOG_GATE_COUNTERS["dropped_ignore_event"] += 1
            return
'''

_OLD_CONDENSE = '''                if not skip_condensing:
                    if any(_ in info for _ in CONDENSE_ON_INFO_KEYWORDS):
                        with _condensing_lock:
                            key = (src_ip, trail)
                            if key not in _condensed_events:
                                _condensed_events[key] = []
                            if len(_condensed_events[key]) < MAX_CONDENSED_EVENTS:
                                _condensed_events[key].append(event_tuple)

                        return
'''

_NEW_CONDENSE = '''                if not skip_condensing:
                    if any(_ in info for _ in CONDENSE_ON_INFO_KEYWORDS):
                        with _condensing_lock:
                            key = (src_ip, trail)
                            if key not in _condensed_events:
                                _condensed_events[key] = []
                            if len(_condensed_events[key]) < MAX_CONDENSED_EVENTS:
                                _condensed_events[key].append(event_tuple)

                        if config.pcap_file:
                            _DIAG_LOG_GATE_COUNTERS["queued_condense"] += 1
                        return
'''

_OLD_THROTTLE = '''                current_bucket = sec // config.PROCESS_COUNT
                if getattr(_thread_data, "log_bucket", None) != current_bucket:  # log throttling
                    _thread_data.log_bucket = current_bucket
                    _thread_data.log_trails = set()
                else:
                    if any(_ in _thread_data.log_trails for _ in ((src_ip, trail), (dst_ip, trail))):
                        return
                    else:
                        _thread_data.log_trails.add((src_ip, trail))
                        _thread_data.log_trails.add((dst_ip, trail))
'''

_NEW_THROTTLE = '''                current_bucket = sec // config.PROCESS_COUNT
                if getattr(_thread_data, "log_bucket", None) != current_bucket:  # log throttling
                    _thread_data.log_bucket = current_bucket
                    _thread_data.log_trails = set()
                else:
                    if any(_ in _thread_data.log_trails for _ in ((src_ip, trail), (dst_ip, trail))):
                        if config.pcap_file:
                            _DIAG_LOG_GATE_COUNTERS["dropped_throttle"] += 1
                        return
                    else:
                        _thread_data.log_trails.add((src_ip, trail))
                        _thread_data.log_trails.add((dst_ip, trail))
'''

_OLD_TAIL = '''                if (config.DISABLE_LOCAL_LOG_STORAGE and not any((config.LOG_SERVER, config.SYSLOG_SERVER))) or config.console:
                    sys.stderr.write(event)
                    sys.stderr.flush()

    except (OSError, IOError):
'''

_NEW_TAIL = '''                if (config.DISABLE_LOCAL_LOG_STORAGE and not any((config.LOG_SERVER, config.SYSLOG_SERVER))) or config.console:
                    sys.stderr.write(event)
                    sys.stderr.flush()

                if config.pcap_file:
                    _DIAG_LOG_GATE_COUNTERS["written"] += 1
            elif config.pcap_file:
                _DIAG_LOG_GATE_COUNTERS["dropped_skip_write"] += 1
        elif config.pcap_file:
            _DIAG_LOG_GATE_COUNTERS["dropped_whitelisted"] += 1

    except (OSError, IOError):
'''

_ALL_PAIRS = ((_OLD_GLOBALS, _NEW_GLOBALS), (_OLD_IGNORE, _NEW_IGNORE), (_OLD_CONDENSE, _NEW_CONDENSE),
              (_OLD_THROTTLE, _NEW_THROTTLE), (_OLD_TAIL, _NEW_TAIL))


def patch_log_diagnostic_gates(log_py_path: Path) -> str:
    """
    Returns one of: "already-patched", "patched", "FAILED-block-not-found".
    """
    text = log_py_path.read_text()
    if _MARKER in text:
        return "already-patched"
    for old, new in _ALL_PAIRS:
        if old not in text:
            return "FAILED-block-not-found"
    for old, new in _ALL_PAIRS:
        text = text.replace(old, new, 1)
    log_py_path.write_text(text)
    return "patched"

# ------------------------------------------------------------------------------------------

"""
Diagnostic patch v3, part 2/2 (temporary, offline-only) -- targets old/sensor.py, completing
part 1/2 (patch_log_diagnostic_gates in core/log.py). Brings _DIAG_LOG_GATE_COUNTERS
from core/log.py into sensor.py so it can be printed as a [diag-v3] line at the same EOF point
as [diag] (v1) and [diag-v2]. Changes NO behaviour at all -- it only reads counters that
core/log.py has already mutated.

SPECIAL PREREQUISITES (different from v1/v2): two things, NOT just this file's own marker --
  1. the target sensor.py MUST already be patched with v2 (marker 'PATCH diagnostic-syn-counters-v2')
     -- the import insertion point lies inside the block injected by v2.
  2. the core/log.py imported at RUNTIME (not at this file's patch time) MUST ALREADY have been
     patched by patch_log_diagnostic_gates (see diag_patcher_v3_log.py) -- otherwise
     sensor.py fails with a clear AttributeError when accessing
     _diag_log_module._DIAG_LOG_GATE_COUNTERS (fail-fast, not silently wrong).
     This is NOT checked by the function below (patch time cannot verify ANOTHER file that is
     run later at runtime) -- the required run order is: core/log.py first, then sensor.py.
"""

_MARKER_V3_SENSOR = "PATCH diagnostic-log-gates-sensor-import"
_V2_MARKER = "PATCH diagnostic-syn-counters-v2"

_OLD_IMPORT = '    return _diag_orig_log_event(event_tuple, *args, **kwargs)\nfrom core import meta\n'

_NEW_IMPORT = "    return _diag_orig_log_event(event_tuple, *args, **kwargs)\n\n# DIAGNOSTIC v3 (temporary, offline-only) -- imports the core.log MODULE (not just one name) so\n# that its _DIAG_LOG_GATE_COUNTERS (see patch_log_diagnostic_gates in core/log.py) keeps pointing\n# to the SAME dict mutated by the original log_event() -- the dict is changed in place (not\n# re-assigned), so this module reference is updated automatically without manual syncing.\n# PREREQUISITE: core/log.py MUST ALREADY be patched (patch_log_diagnostic_gates) BEFORE this\n# sensor.py is imported/run -- otherwise the line below fails with a clear AttributeError (rather\n# than silently miscounting), because _DIAG_LOG_GATE_COUNTERS does not exist in the module yet.\n# Marker: 'PATCH diagnostic-log-gates-sensor-import'.\nfrom core import log as _diag_log_module\nfrom core import meta\n"

_OLD_PRINT = '                    elif config.pcap_file:\n                        with _done_lock:\n                            _done_count += 1\n                        c = _DIAG_SYN_COUNTERS\n                        print("[diag] SYN counters for \'%s\': total=%d burst_skipped=%d trail_hit=%d "\n                              "logged_skipped=%d alert_emitted=%d" % (config.pcap_file, c["total"],\n                              c["burst_skipped"], c["trail_hit"], c["logged_skipped"], c["alert_emitted"]))\n                        print("[diag-v2] would-match (pre-burst-skip) for \'%s\': would_match_total=%d "\n                              "would_match_dst=%d would_match_src=%d would_match_via_addr_port=%d "\n                              "burst_skipped_would_have_matched=%d" % (config.pcap_file,\n                              c["would_match_total"], c["would_match_dst"], c["would_match_src"],\n                              c["would_match_via_addr_port"], c["burst_skipped_would_have_matched"]))\n                        if _DIAG_LOG_EVENT_CALLS:\n                            print("[diag-v2] log_event calls for \'%s\': %s" % (config.pcap_file,\n                                  ", ".join("%s=%d" % (k, v) for k, v in sorted(_DIAG_LOG_EVENT_CALLS.items()))))\n                        else:\n                            print("[diag-v2] log_event calls for \'%s\': (no calls at all)" % config.pcap_file)\n                        break\n'

_NEW_PRINT = '                    elif config.pcap_file:\n                        with _done_lock:\n                            _done_count += 1\n                        c = _DIAG_SYN_COUNTERS\n                        print("[diag] SYN counters for \'%s\': total=%d burst_skipped=%d trail_hit=%d "\n                              "logged_skipped=%d alert_emitted=%d" % (config.pcap_file, c["total"],\n                              c["burst_skipped"], c["trail_hit"], c["logged_skipped"], c["alert_emitted"]))\n                        print("[diag-v2] would-match (pre-burst-skip) for \'%s\': would_match_total=%d "\n                              "would_match_dst=%d would_match_src=%d would_match_via_addr_port=%d "\n                              "burst_skipped_would_have_matched=%d" % (config.pcap_file,\n                              c["would_match_total"], c["would_match_dst"], c["would_match_src"],\n                              c["would_match_via_addr_port"], c["burst_skipped_would_have_matched"]))\n                        if _DIAG_LOG_EVENT_CALLS:\n                            print("[diag-v2] log_event calls for \'%s\': %s" % (config.pcap_file,\n                                  ", ".join("%s=%d" % (k, v) for k, v in sorted(_DIAG_LOG_EVENT_CALLS.items()))))\n                        else:\n                            print("[diag-v2] log_event calls for \'%s\': (no calls at all)" % config.pcap_file)\n                        if _diag_log_module._DIAG_LOG_GATE_COUNTERS["total"]:\n                            g = _diag_log_module._DIAG_LOG_GATE_COUNTERS\n                            print("[diag-v3] log_event internal gates for \'%s\': total=%d "\n                                  "dropped_ignore_event=%d dropped_whitelisted=%d dropped_skip_write=%d "\n                                  "queued_condense=%d dropped_throttle=%d written=%d" % (config.pcap_file,\n                                  g["total"], g["dropped_ignore_event"], g["dropped_whitelisted"],\n                                  g["dropped_skip_write"], g["queued_condense"], g["dropped_throttle"],\n                                  g["written"]))\n                        else:\n                            print("[diag-v3] log_event internal gates for \'%s\': (log_event was never called at all)" % config.pcap_file)\n                        break\n'

_ALL_PAIRS_V3_SENSOR = ((_OLD_IMPORT, _NEW_IMPORT), (_OLD_PRINT, _NEW_PRINT))


def patch_sensor_diagnostic_v3(sensor_py_path: Path) -> str:
    """
    Returns one of: "already-patched", "patched", "FAILED-v2-prerequisite-missing",
    "FAILED-block-not-found".
    """
    text = sensor_py_path.read_text()
    if _MARKER_V3_SENSOR in text:
        return "already-patched"
    if _V2_MARKER not in text:
        return "FAILED-v2-prerequisite-missing"
    for old, new in _ALL_PAIRS_V3_SENSOR:
        if old not in text:
            return "FAILED-block-not-found"
    for old, new in _ALL_PAIRS_V3_SENSOR:
        text = text.replace(old, new, 1)
    sensor_py_path.write_text(text)
    return "patched"

CORE_LOG_PY = MALTRAIL_DIR / "core" / "log.py"
print(patch_log_diagnostic_gates(CORE_LOG_PY))
print(patch_sensor_diagnostic_v3(SENSOR_PY))


### C.2 — Build the local config (`maltrail.local.conf`)

The original `maltrail.conf` has two things that need overriding for this experiment, without changing the original file:
- `LOG_DIR` — defaults to `$SYSTEM_LOG_DIR/maltrail` (a system variable, usually requiring root access to a path such as `/var/log/maltrail`). Redirected to `data/maltrail_out/` inside this experiment folder.
- `DISABLE_CHECK_SUDO true` — without it, `old/sensor.py` refuses to run (`please run ... with root privileges`) even in `-r` replay mode, which technically needs no raw-socket capture privilege at all (confirmed: the check is *unconditional*, making no distinction between live capture and offline replay, in `core/log.py`). This option is **officially supported upstream** (used exactly this way in their own test suite, `core/testing.py`) — not an ad-hoc patch/bypass, and deliberately chosen over `sudo` for least privilege (the packet parser runs without root privileges).

- `PROCESS_COUNT 1` — **added 25 August 2026** after investigating the Zeus alert-count drift (see README.md "Investigation of the Zeus alert-count drift"). The upstream `maltrail.conf` template sets `PROCESS_COUNT 16` (16 parallel processes, round-robin without `USE_CAPTURE_AFFINITY`). Combined with the pcapy-ng bug that makes `old/sensor.py` replace the original pcap timestamps with the process wall clock, this combination was shown to make the time-windowed heuristics (scan/DNS exhaustion/dedup) non-deterministic across runs on a byte-identical pcap (16→13→13→18 Zeus alerts over 4 runs). `PROCESS_COUNT 1` removes the multiprocessing layer of non-determinism — its empirical verification is in the new cell C.5 below.

In [ ]:
def build_maltrail_local_conf() -> None:
    base_conf = MALTRAIL_DIR / "maltrail.conf"
    content = base_conf.read_text()
    abs_log_dir = str(MALTRAIL_OUT_DIR.resolve())

    new_lines = []
    replaced_log_dir = False
    replaced_process_count = False
    for line in content.splitlines():
        if line.strip().startswith("LOG_DIR "):
            new_lines.append(f"LOG_DIR {abs_log_dir}")
            replaced_log_dir = True
        elif line.strip().startswith("PROCESS_COUNT "):
            # Override of 25 August 2026 -- see README.md "Investigation of the Zeus alert-count drift".
            # The upstream template sets PROCESS_COUNT 16 (round-robin multiprocessing), which,
            # combined with the pcapy-ng bug (wall clock replacing the original pcap timestamps),
            # was shown to make the time-windowed heuristics non-deterministic across runs on a
            # byte-identical pcap. PROCESS_COUNT 1 removes the multiprocessing layer of non-determinism
            # -- see verification cell C.5 below for the empirical test.
            new_lines.append("PROCESS_COUNT 1")
            replaced_process_count = True
        else:
            new_lines.append(line)
    if not replaced_log_dir:
        new_lines.append(f"LOG_DIR {abs_log_dir}")
    if not replaced_process_count:
        new_lines.append("PROCESS_COUNT 1")

    new_lines.append("")
    new_lines.append("# --- automatic override by layer1_experiment.ipynb, see Section C.2 ---")
    new_lines.append("DISABLE_CHECK_SUDO true")

    MALTRAIL_LOCAL_CONF.write_text("\n".join(new_lines) + "\n")
    print("Written:", MALTRAIL_LOCAL_CONF)
    print("LOG_DIR pointed to:", abs_log_dir)
    print("PROCESS_COUNT locked to 1 (for offline-replay reproducibility -- see README.md).")


if MALTRAIL_DIR.exists():
    build_maltrail_local_conf()

### C.2a — Pin `trails.csv` (cross-session reproducibility, 28 August 2026)

**Context**: the 28 August C.5 run (after the tinba fix) shows `tinba.pcap` PERFECTLY deterministic
within one execution (5x10902 identical) — but comparing two SEPARATE runs on the same day, the total
`log_event` calls (`[diag-v3]`) for tinba changed from **10827** (01:55 run, before the timestamp
fix) to **10902** (03:39 run, after this fix). This is a different class from the non-determinism
already fixed (which was within-session) — this is about reproducibility ACROSS sessions.

**Checked directly** [HIGH CONFIDENCE — live mtime & hash taken directly from `~/.maltrail/` on the
Mac, not a guess]: `trails.csv` **HAS in fact changed** since the earlier frozen snapshot (24
August, `trails_2026-08-24T1443Z_frozen.csv`, 92,096,534 bytes) — the live mtime is now
`2026-08-28T01:52:20Z`, with the size grown to 93,130,085 bytes. **This does NOT contradict** the
earlier README conclusion ("the trails.csv drift hypothesis was PROVEN WRONG", 24–25 August) — that
conclusion was specific to the time window tested then (mtime shown to be stable DURING the 25
August test session), not a claim that trails.csv never changes at all. Now, >3 days later,
Maltrail's `UPDATE_PERIOD` (24 hours) has triggered a refresh at least once.

[MEDIUM-HIGH CONFIDENCE, **not yet verified to the minute**] this trails.csv refresh (01:52:20Z)
likely explains the difference total=10827 vs total=10902 for tinba — but it cannot be established
exactly which run (01:55 or 03:39) used which version of trails.csv. This **needs verification** if
the exact figure is wanted; it does not block the fix below, which removes this whole class of
problem going forward regardless of the exact cause this time.

**Fix**: two functions — `freeze_trails_snapshot()` freezes the current LIVE `trails.csv` +
`trails.csv.confidence` into `trails_snapshots/`, with a manifest (mtime, size, SHA-256) following
the format of the manual 24 August snapshot already in this project.
`pin_trails_from_snapshot()` restores that frozen snapshot BACK into `~/.maltrail/` before an
experiment run, verifies SHA-256 **before AND after** the restore (not assumed), and deletes
`trails.csv.bin`/`.bin.lock` (Maltrail's compiled binary cache of trails.csv) — if the old cache
were left in place, Maltrail might silently keep using a cache that does NOT match the newly
pinned trails.csv, making the pinning invisibly useless (the same class of bug as the earlier
marker-not-embedded bug: it looks successful while not actually changing what Maltrail reads).

**Baseline used**: the current live `trails.csv` (mtime `2026-08-28T01:52:20Z`, SHA-256
`6b7d9e7b96af061...` — see the full manifest) has been frozen to
`trails_snapshots/trails_2026-08-28T0152Z_frozen.csv[.confidence]` +
`MANIFEST_trails_2026-08-28T0152Z.txt`. This is likely the same trails.csv used by the 03:39 run
(the zeus/tinba deterministic C.5 results just confirmed) — chosen as the first pinned baseline so
that those already-validated figures do not change again because of the pinning itself.
The cell below locks `PINNED_TRAILS_STAMP` to this stamp explicitly (not an automatic pick of the
"latest") — if a new version needs freezing later (e.g. a new trail feed relevant to additional
pcaps), call `freeze_trails_snapshot()` manually and update `PINNED_TRAILS_STAMP` deliberately,
recording here when & why.

Tested before delivery — 10 isolated scenarios (fixtures, not the real trails.csv): freeze fails
cleanly if `~/.maltrail` does not exist / the source files are incomplete; a successful freeze
produces a manifest + a byte-identical copy (SHA-256 re-checked, not assumed); pin fails cleanly if
the manifest is missing; **freeze→drift→pin roundtrip** — the live file deliberately changed
(simulated "drift" refresh), pin restores it EXACTLY to the frozen content, old
`trails.csv.bin`/`.bin.lock` deleted; idempotency (pin twice in a row, both succeed, same end
result); **two fail-safe paths** tested explicitly — the frozen snapshot missing from disk, and
the frozen snapshot CORRUPT (hash does not match the manifest) — both fail cleanly WITHOUT touching
the live trails.csv at all (confirmed by asserting the live content is unchanged);
`.bin`/`.bin.lock` not existing at all (fresh install) does not crash when deletion is attempted;
the manifest records the correct byte size; the manifest parser was also tested against the REAL
manifest text written manually on the Mac (including extra narrative after the sha256 block,
ensuring it is not wrongly parsed).

**Not yet verified in a real run of this notebook** — the cell below calls
`pin_trails_from_snapshot()` for real against `~/.maltrail/` when run. If it fails (`FAILED-...`),
the cell raises an exception and STOPS execution — deliberately, so that C.3/C.5 do not silently
run with unpinned trail data.


**UPDATE (22 October 2026) — bug found & fixed: this pin was only effective for 24 hours SINCE THE
SNAPSHOT WAS FROZEN, not 24 hours since it was pinned.** `pin_trails_from_snapshot()` above uses
`shutil.copy2()`, which deliberately preserves the mtime of the source file (the frozen snapshot).
So if the experiment is run >24 hours after the SNAPSHOT was made (not after it was last pinned),
Maltrail's `UPDATE_PERIOD` still considers `trails.csv` stale and rebuilds its content itself
from static trails only (without feeds) right after the pin "succeeded" -- the pinning becomes
useless without any sign of failure.

**Found** [HIGH CONFIDENCE, direct evidence] while running Maltrail on
`ctu_normal33.pcap` (22 October 2026, >55 days after snapshot `2026-08-28T0152Z` was frozen):
`trails.csv` changed content DURING a single `--offline` run -- SHA-256 `6b7d9e7b96af061...` became
`7cdabd0b...`, the line count dropped from 1,791,806 to 1,632,628 (consistent with a rebuild
without feeds, leaving only static trails).

**This does NOT contradict** the 26 August conclusion above ("the trails.csv drift hypothesis was
PROVEN WRONG") -- that conclusion was specific to a staleness of ~34 hours (the 25 August test
session), and was correct for that range: trails.csv indeed did not change within 34 hours. The
22 October finding is the same mechanism (`UPDATE_PERIOD` + rebuild-without-feeds in
`core/update.py::update_trails`) but at a far more extreme staleness (55 days) never tested before --
the two are consistent, not contradictory.

**Fix**: `pin_trails_from_snapshot()` now calls `.touch()` on both files after `copy2()`,
refreshing the mtime to the PIN time (not the time the snapshot was frozen) without changing the
content (byte-identical, still verified by SHA-256 after pinning as before). The 24-hour window is
now counted from when the files actually start being used, not from when their historical snapshot
was made. The same function (standalone, `tools/pin_trails_fresh.py`) was used & verified directly
today -- the Tinba positive control (re-run with the fixed pin) produced 10,902 alerts,
byte-identical with the 28 August `tinba.log`.

In [ ]:
"""
Pin trails.csv (28 August 2026) -- see markdown C.2a above for context & test results.
Copies the frozen trails.csv/.confidence from trails_snapshots/ BACK into ~/.maltrail/ before each
experiment run, deleting trails.csv.bin/.bin.lock so that Maltrail must rebuild its cache from
the newly pinned trails.csv (otherwise the old cache could silently keep being used).
"""
from __future__ import annotations

import hashlib
import shutil
from datetime import datetime, timezone
from pathlib import Path

TRAILS_FILES = ("trails.csv", "trails.csv.confidence")
_BIN_CACHE_FILES = ("trails.csv.bin", "trails.csv.bin.lock")

MALTRAIL_HOME = Path.home() / ".maltrail"
TRAILS_SNAPSHOTS_DIR = BASE_DIR / "trails_snapshots"


def _sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def _utc_compact_stamp(dt: datetime) -> str:
    return dt.strftime("%Y-%m-%dT%H%MZ")


def freeze_trails_snapshot(maltrail_home: Path, snapshot_dir: Path) -> str:
    """Freezes the LIVE trails.csv + trails.csv.confidence into snapshot_dir. Call manually when
    a new pin baseline is needed -- see markdown C.2a."""
    if not maltrail_home.exists():
        return "FAILED-maltrail-home-missing"

    src_paths = {name: maltrail_home / name for name in TRAILS_FILES}
    missing = [name for name, p in src_paths.items() if not p.exists()]
    if missing:
        return "FAILED-source-files-incomplete"

    src_mtime = datetime.fromtimestamp(src_paths["trails.csv"].stat().st_mtime, tz=timezone.utc)
    stamp = _utc_compact_stamp(src_mtime)
    snapshot_dir.mkdir(parents=True, exist_ok=True)

    src_hashes = {name: _sha256(p) for name, p in src_paths.items()}
    src_sizes = {name: p.stat().st_size for name, p in src_paths.items()}

    dest_paths = {}
    for name, p in src_paths.items():
        suffix = name[len("trails.csv") :]
        dest = snapshot_dir / f"trails_{stamp}_frozen.csv{suffix}"
        shutil.copy2(p, dest)
        dest_paths[name] = dest

    dest_hashes = {name: _sha256(p) for name, p in dest_paths.items()}
    for name in TRAILS_FILES:
        if dest_hashes[name] != src_hashes[name]:
            return "FAILED-copy-hash-verification-mismatch"

    manifest_path = snapshot_dir / f"MANIFEST_trails_{stamp}.txt"
    frozen_at = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    src_mtime_str = src_mtime.strftime("%Y-%m-%dT%H:%M:%SZ")
    lines = [
        "# Snapshot of trails.csv frozen for cross-session determinism pinning",
        f"frozen_at_utc: {frozen_at}",
        f"source: {maltrail_home}/trails.csv (+ trails.csv.confidence)",
        f"source_mtime_utc: {src_mtime_str}",
        "source_size_bytes:",
        f"  trails.csv: {src_sizes['trails.csv']}",
        f"  trails.csv.confidence: {src_sizes['trails.csv.confidence']}",
        "sha256:",
        f"  trails.csv:             {src_hashes['trails.csv']}",
        f"  trails.csv.confidence:  {src_hashes['trails.csv.confidence']}",
        "",
    ]
    manifest_path.write_text("\n".join(lines))
    return f"frozen:{manifest_path}"


def _parse_manifest(manifest_path: Path) -> dict:
    text = manifest_path.read_text()
    out = {"sha256": {}, "size": {}}
    section = None
    for raw in text.splitlines():
        if raw.strip() == "source_size_bytes:":
            section = "size"
            continue
        if raw.strip() == "sha256:":
            section = "sha256"
            continue
        if section and raw.startswith("  ") and ":" in raw:
            key, val = raw.strip().split(":", 1)
            key = key.strip()
            val = val.strip()
            if key in TRAILS_FILES:
                out[section][key] = int(val) if section == "size" else val
        elif raw and not raw.startswith(" "):
            section = None
    return out


def pin_trails_from_snapshot(maltrail_home: Path, manifest_path: Path, snapshot_dir: Path) -> str:
    """Restores the frozen snapshot (referenced by manifest_path) into maltrail_home. See markdown
    C.2a for the list of failure codes & the reason for each verification."""
    if not manifest_path.exists():
        return "FAILED-manifest-missing"

    manifest = _parse_manifest(manifest_path)
    if set(manifest["sha256"]) != set(TRAILS_FILES):
        return "FAILED-manifest-missing"

    stamp = manifest_path.stem.replace("MANIFEST_trails_", "")
    snapshot_paths = {}
    for name in TRAILS_FILES:
        suffix = name[len("trails.csv") :]
        snapshot_paths[name] = snapshot_dir / f"trails_{stamp}_frozen.csv{suffix}"

    if any(not p.exists() for p in snapshot_paths.values()):
        return "FAILED-snapshot-missing"

    for name, p in snapshot_paths.items():
        if _sha256(p) != manifest["sha256"][name]:
            return "FAILED-snapshot-corrupt"

    if not maltrail_home.exists():
        return "FAILED-maltrail-home-missing"

    for name, p in snapshot_paths.items():
        shutil.copy2(p, maltrail_home / name)

    # FIX (22 October 2026, see README.md "trails.csv pin bug found & fixed"):
    # shutil.copy2() above DELIBERATELY preserves the mtime of the original snapshot file (that is
    # what copy2 does -- it copies metadata including mtime). As a result this pin was only
    # effective for 24 hours (UPDATE_PERIOD) since the snapshot was frozen, NOT 24 hours since it
    # was pinned -- if the experiment runs >24 hours after the snapshot was made, Maltrail silently
    # rebuilds trails.csv from static trails only (without feeds) right after this pin "succeeds",
    # and the pin becomes useless without visibly failing. Found 22 October 2026 (first CTU-Normal-33
    # run: trails.csv changed content DURING a single offline run, sha256 6b7d9e7b...->7cdabd0b...),
    # fixed here with touch() -- the content stays byte-identical (verified by sha256 below), only
    # the mtime is refreshed to the PIN time (not the time the snapshot was frozen), so that the
    # 24-hour window is counted from when this file actually starts being used.
    for name in TRAILS_FILES:
        (maltrail_home / name).touch()

    for cache_name in _BIN_CACHE_FILES:
        cache_path = maltrail_home / cache_name
        if cache_path.exists():
            cache_path.unlink()

    for name in TRAILS_FILES:
        if _sha256(maltrail_home / name) != manifest["sha256"][name]:
            return "FAILED-post-pin-verification-mismatch"

    return "pinned"


# --- Apply the pin ---
# Baseline chosen explicitly (28 August 2026) -- see markdown C.2a for why this stamp was
# chosen. Update it DELIBERATELY (not an automatic pick of the "latest") if a new snapshot is
# made later via freeze_trails_snapshot(MALTRAIL_HOME, TRAILS_SNAPSHOTS_DIR).
PINNED_TRAILS_STAMP = "2026-08-28T0152Z"

_manifest_path = TRAILS_SNAPSHOTS_DIR / f"MANIFEST_trails_{PINNED_TRAILS_STAMP}.txt"
_pin_result = pin_trails_from_snapshot(MALTRAIL_HOME, _manifest_path, TRAILS_SNAPSHOTS_DIR)
print("Pin trails.csv:", _pin_result)

if _pin_result == "pinned":
    print("  live trails.csv locked to snapshot", PINNED_TRAILS_STAMP)
    print("  live SHA-256 now:", _sha256(MALTRAIL_HOME / "trails.csv"))
else:
    raise RuntimeError(
        f"Pinning trails.csv FAILED ({_pin_result}) -- do NOT continue to C.3/C.5, the alert counts "
        f"will not be reproducible. See README.md 'trails.csv pin bug found & fixed' and the manifest at {_manifest_path}."
    )


### C.3 — Run the sensor per pcap

Important details already verified (including one bug caught by the first run of this notebook on the Mac — see the last point):
- The command is run with `cwd=MALTRAIL_DIR` (not `BASE_DIR`) — because `CUSTOM_TRAILS_DIR` in the config is a **relative** path (`./trails/custom`), it must resolve relative to the `tools/maltrail` folder, not this notebook's folder.
- Maltrail log files are named by **date**, not per run (`LOG_DIR/<YYYY-MM-DD>.log`, from `get_event_log_handle(sec)` in `core/log.py`) — if not moved right after each run, the next pcap's results can pile into the same file.
- **IMPORTANT CORRECTION (26 August 2026) — the "empirical" claim in the previous bullet was PROVEN WRONG, see README.md "Log-file discovery bug v3"**: this place previously said "if there are zero alerts, Maltrail does not create a log file at all — empirically confirmed". That was WRONG. What actually happens: the date in the file name comes from `sec` (the timestamp), and since the offline-timestamp patch (Section C.1b), `sec` in offline mode = the ORIGINAL pcap timestamp, NOT today. For a pcap whose original capture date is not today (e.g. `zeus.pcap` = 2012-05-29), Maltrail STILL writes alerts — just to the file `2012-05-29.log`, not the `<today>.log` the old code below looked for. The "zero alerts" seen earlier were a WRONG FILE LOOKUP, not a true negative detection — proven by the C.1e instrumentation (`written=5696` for zeus, not zero) and by the file `2012-05-29.log` that did in fact exist on disk (205,092 real alert lines). The code below now detects which file ACTUALLY grew during each run (a byte-size snapshot before/after), rather than guessing the file name from any date. **Impact on earlier conclusions**: the C.5 determinism conclusion ("IDENTICAL over 5 runs, PROVEN SUFFICIENT") obtained BEFORE this fix is also invalid — its numbers were `[0, 0, 0, 0, 0]`, literally identical but because all five equally failed to find the file, not because the pipeline was truly reproducible. C.5 has been fixed with the same mechanism; it must be run AGAIN for a valid determinism conclusion.
- The trail set (`~/.maltrail/trails.csv`) is re-fetched from the internet ONLY if the file does not exist yet or is older than `UPDATE_PERIOD` (default 86400 seconds/24 hours in `maltrail.conf`) — so this notebook can be re-run many times without repeated fetching while the trails are still fresh, BUT all pilot pcaps within one batch are still evaluated against the same trail snapshot (methodological consistency).
- **BUG found & fixed**: an earlier version of this notebook called `sensor.py` via `sys.executable` — it turned out that in the Jupyter/VS Code kernel used, `sys.executable` pointed to `/Library/Developer/CommandLineTools/usr/bin/python3` (the system Python), **not** the `.venv` where `pcapy-ng` is installed — even though the `.venv` kernel had been selected in VS Code. As a result ALL runs failed (`returncode=1`, stderr `please install 'pcapy or pcapy-ng'`) before touching the trails/pcap at all, and this looked like a misleading "0 alerts" (not a negative detection result but an execution failure). The code below now explicitly looks for `.venv/bin/python3` in this notebook's folder first, plus an `import pcapy` pre-check before the main loop runs — so that if a similar problem happens again, the error is clear BEFORE the 5 runs are attempted, rather than hidden as "0 alerts" in all five.


In [ ]:
import re, time

TRAILS_CSV = Path.home() / ".maltrail" / "trails.csv"
TRAILS_MAX_AGE_HOURS = 24  # aligned with the default UPDATE_PERIOD (86400 seconds) in maltrail.conf


def trails_are_fresh() -> bool:
    if not TRAILS_CSV.exists():
        return False
    age_hours = (time.time() - TRAILS_CSV.stat().st_mtime) / 3600
    return age_hours < TRAILS_MAX_AGE_HOURS


def resolve_maltrail_python() -> str:
    '''Explicitly look for the python of this experiment folder's .venv -- do NOT simply rely on
    sys.executable, since it was shown that it can wrongly point to the system Python
    even when the .venv kernel is selected in VS Code (see the bug note above).'''
    venv_python = BASE_DIR / ".venv" / "bin" / "python3"
    if venv_python.exists():
        return str(venv_python)
    print(f"[!] {venv_python} not found -> falling back to sys.executable ({sys.executable}).")
    print("    Check manually: does .venv actually exist in this folder, with pcapy-ng installed in it?")
    return sys.executable


def check_pcapy_available(python_exe: str) -> bool:
    r = subprocess.run([python_exe, "-c", "import pcapy; print(pcapy.__file__)"],
                        capture_output=True, text=True, timeout=30)
    if r.returncode == 0:
        print(f"[OK] pcapy can be imported via {python_exe}\n     -> {r.stdout.strip()}")
        return True
    print(f"[!] pcapy can NOT be imported via {python_exe}")
    print("    stderr:", r.stderr.strip()[:300])
    print("    Fix this first (e.g. `source .venv/bin/activate && pip install pcapy-ng`) before continuing.")
    return False


# --- Fix of 26 August 2026 (see README.md "Log-file discovery bug v3") ----------------------
# `get_event_log_handle(sec)` in core/log.py WRITES to `<LOG_DIR>/<date from sec>.log` --
# NOT `<today>.log`. Since Fix #2 (offline-timestamp), `sec` = the ORIGINAL pcap timestamp in
# offline mode -- so a pcap whose original capture date is NOT today (e.g. zeus.pcap = 2012-05-29)
# is written to a file the old function NEVER looked for. This is NOT Maltrail failing to detect --
# proven by the C.1e instrumentation (`written=5696` for zeus) and by the file `2012-05-29.log`
# that ALREADY EXISTED on disk, containing 205,092 real alert lines (blackshades etc.).
#
# Fix: do NOT guess the file name from a date (today OR from sec) -- detect which file
# ACTUALLY grew DURING this subprocess run, via a byte-size snapshot
# before/after. Maltrail log files are ALWAYS opened in append mode (O_APPEND), so the slice
# byte[size_before:] of a grown file = EXACTLY this run's contribution -- even if that file
# has already accumulated thousands of lines from earlier runs (like `2012-05-29.log` now).
DATE_LOG_RE = re.compile(r"^\d{4}-\d{2}-\d{2}\.log$")


def snapshot_dated_logs() -> dict:
    """Size (bytes) of each <YYYY-MM-DD>.log file that EXISTS NOW in MALTRAIL_OUT_DIR -- called
    BEFORE the subprocess runs, so it is the baseline for measuring the delta afterwards."""
    retval = {}
    if MALTRAIL_OUT_DIR.exists():
        for p in MALTRAIL_OUT_DIR.iterdir():
            if p.is_file() and DATE_LOG_RE.match(p.name):
                retval[p.name] = p.stat().st_size
    return retval


def collect_new_log_lines(before: dict) -> tuple:
    """Compare the snapshot BEFORE with the CURRENT state; return (the combined bytes NEWLY written
    by this run, the list of touched file names). Tested in isolation (9 scenarios, see README.md) before
    being used here -- including a large old file + small delta (exactly the real zeus case), two
    files touched at once (a pcap spanning midnight / several pcaps on the same wall-clock
    fallback), and a file that shrank since the snapshot (taken IN FULL, not ignored)."""
    if not MALTRAIL_OUT_DIR.exists():
        return b"", []
    touched, chunks = [], []
    for p in sorted(MALTRAIL_OUT_DIR.iterdir()):
        if not (p.is_file() and DATE_LOG_RE.match(p.name)):
            continue
        before_size = before.get(p.name, 0)
        after_size = p.stat().st_size
        if after_size > before_size:
            touched.append(p.name)
            with open(p, "rb") as f:
                f.seek(before_size)
                chunks.append(f.read())
        elif after_size < before_size:
            touched.append(f"{p.name} (shrank since snapshot -- taken in full)")
            chunks.append(p.read_bytes())
    return b"".join(chunks), touched


def run_maltrail_on_pcap(python_exe: str, pcap_path: Path, offline: bool, timeout_s: int = 300) -> subprocess.CompletedProcess:
    cmd = [
        python_exe, str(SENSOR_PY),
        "-r", str(pcap_path.resolve()),
        "-c", "maltrail.local.conf",
        "--console",
    ]
    if offline:
        cmd.append("--offline")
    print(f"Running (cwd={MALTRAIL_DIR}):", " ".join(cmd))
    return subprocess.run(cmd, capture_output=True, text=True, cwd=MALTRAIL_DIR, timeout=timeout_s)


maltrail_results = {}
if MALTRAIL_DIR.exists() and SENSOR_PY.exists():
    MALTRAIL_PYTHON = resolve_maltrail_python()
    if check_pcapy_available(MALTRAIL_PYTHON):
        trails_fresh_at_start = trails_are_fresh()
        print(f"trails.csv fresh (< {TRAILS_MAX_AGE_HOURS} hours)? {trails_fresh_at_start}")
        print()
        for i, (key, pcap_path) in enumerate(downloaded.items()):
            # fetch the real trails only if not fresh AND this is the first run in this batch
            need_fetch = (i == 0) and not trails_fresh_at_start
            _before_snapshot = snapshot_dated_logs()
            proc = run_maltrail_on_pcap(MALTRAIL_PYTHON, pcap_path, offline=not need_fetch)
            _new_bytes, _touched_files = collect_new_log_lines(_before_snapshot)

            dest_log = MALTRAIL_OUT_DIR / f"{key}.log"
            if _new_bytes:
                dest_log.write_bytes(_new_bytes)
                n_lines = _new_bytes.count(b"\n")
            else:
                dest_log, n_lines = None, 0

            maltrail_results[key] = {"returncode": proc.returncode, "log_path": dest_log, "n_lines": n_lines}
            tag = "WITHOUT --offline (fetching real trails)" if need_fetch else "--offline (using trail snapshot)"
            print(f"[{key}] {tag} -> returncode={proc.returncode}, alert lines={n_lines}"
                  + (f", log={dest_log.name} (from: {', '.join(_touched_files)})" if dest_log
                     else " (no dated log file grew -> 0 alerts)"))
            for _diag_line in proc.stdout.splitlines():
                if _diag_line.startswith("[diag"):  # catches [diag] (v1), [diag-v2], and [diag-v3]
                    print("   ", _diag_line)
            if proc.returncode != 0:
                print("   stderr:", proc.stderr.strip()[:500])
    else:
        print("pcapy pre-check failed -> main loop skipped so as not to produce 5 misleading 'failed to run' results.")
else:
    print("MALTRAIL_DIR/SENSOR_PY incomplete -> this cell is skipped. Check cell C.1 above.")


### C.4 — Parse the Maltrail log into pandas

Format of one log line (from direct observation of real runs, NOT official documentation — Maltrail does not formally document this format):

```
"<timestamp>" <hostname> <src_ip> <src_port> <dst_ip> <dst_port> <proto> <event_type> <observable> "<label> (<severity>)"(<trail_category>)
```

Two quirks found directly from real data (not assumed up front):
1. The space before `(<trail_category>)` is **inconsistent** — sometimes it sticks directly to the closing quote of the label, sometimes there is one space. The regex below uses `\s*` (zero or more spaces) to tolerate both variants.
2. The `observable` field is usually a token without spaces (domain/IP), BUT for events of type `UA` (User-Agent), the observable itself is quoted and may contain spaces/parentheses (e.g. `"NSIS_Inetc( \(Mozilla\))"`) — the regex below handles both forms (quoted or not).

3. `trail_category` (the very end of the line) is usually bare parentheses, e.g. `(static)` or `(heuristic)`. BUT a second form was found on 25 August 2026 from the "tor exit node" feed (IoT-23 pcap): QUOTED, containing a multi-source citation, e.g. `"torproject.org (+abuseipdb.com,dataplane.org,sblam.com)"` -- its content may itself contain parentheses & commas. The regex below now handles BOTH forms (tested with a 4-case fixture before delivery, including original lines from a real run).

Lines that do not match the regex at all are **still kept** (other columns empty, `_raw_line` filled) so that no data is silently lost — if this happens, it is a signal that the format has changed again and needs to be reported.

In [ ]:
import re
import pandas as pd  # used here and again in Section D -- safe to import twice

# trail_category has TWO forms found in real data:
#   1. bare parentheses, e.g. (static) or (heuristic) -- the most common case
#   2. quoted, containing a multi-source citation, e.g.
#      "torproject.org (+abuseipdb.com,dataplane.org,sblam.com)" -- found 25 August 2026
#      from the "tor exit node" feed; its content may itself contain parentheses & commas.
# Python `re` does not allow the same group name to be repeated across alternatives, so two
# separate groups (_paren / _quoted) are merged into one "trail_category" column in parse_maltrail_log().
MALTRAIL_LOG_RE = re.compile(
    r'^"(?P<timestamp>[^"]+)"\s+(?P<hostname>\S+)\s+(?P<src_ip>\S+)\s+(?P<src_port>\d+)\s+'
    r'(?P<dst_ip>\S+)\s+(?P<dst_port>\d+)\s+(?P<proto>\S+)\s+(?P<event_type>\S+)\s+'
    r'(?P<observable>"[^"]*"|\S+)\s+"(?P<label>[^"]+)"\s*'
    r'(?:\((?P<trail_category_paren>[^)]+)\)|"(?P<trail_category_quoted>[^"]+)")\s*$'
)

MALTRAIL_COLUMNS = ["timestamp", "hostname", "src_ip", "src_port", "dst_ip", "dst_port",
                     "proto", "event_type", "observable", "label", "trail_category", "_raw_line"]


def parse_maltrail_log(log_path: Optional[Path]) -> pd.DataFrame:
    if log_path is None or not log_path.exists():
        return pd.DataFrame(columns=MALTRAIL_COLUMNS)

    rows, unmatched_lines = [], []
    with open(log_path, encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line.strip():
                continue
            m = MALTRAIL_LOG_RE.match(line)
            if m:
                d = m.groupdict()
                d["observable"] = d["observable"].strip('"')
                # merge the two trail_category forms (see the comment above the regex) into one column
                tc_paren = d.pop("trail_category_paren")
                tc_quoted = d.pop("trail_category_quoted")
                d["trail_category"] = tc_paren if tc_paren is not None else tc_quoted
                d["_raw_line"] = line
                rows.append(d)
            else:
                unmatched_lines.append(line)
                rows.append({col: None for col in MALTRAIL_COLUMNS[:-1]} | {"_raw_line": line})

    if unmatched_lines:
        print(f"   [!] {len(unmatched_lines)} lines do NOT MATCH the regex in {log_path.name} -- raw content:")
        for raw in unmatched_lines[:5]:   # show directly, do not rely on head(10) below
            print(f"       {raw!r}")
        if len(unmatched_lines) > 5:
            print(f"       ... and {len(unmatched_lines) - 5} more lines (check the _raw_line column in the dataframe).")
    return pd.DataFrame(rows, columns=MALTRAIL_COLUMNS)


maltrail_frames, maltrail_summary = [], {}
for key, info in maltrail_results.items():
    df = parse_maltrail_log(info["log_path"])
    df["pilot_key"] = key
    maltrail_frames.append(df)
    n_lines = len(df)
    n_unique = int(df["observable"].nunique(dropna=True)) if not df.empty else 0
    maltrail_summary[key] = {"n_alert_lines": n_lines, "n_unique_observables": n_unique}
    print(f"[{key}] alert lines={n_lines}, unique observables={n_unique}")

all_maltrail = pd.concat(maltrail_frames, ignore_index=True) if maltrail_frames else pd.DataFrame(columns=MALTRAIL_COLUMNS)
if not all_maltrail.empty:
    display(all_maltrail.head(10))
else:
    print("No Maltrail alerts have been parsed yet (check maltrail_results above).")

### C.5 — Determinism verification (drift investigation only, updated 28 August 2026)

**CORRECTION #1 (26 August 2026)**: the earlier run reported `All runs: [0, 0, 0, 0, 0]` -> "RESULT:
IDENTICAL ... PROVEN SUFFICIENT, safe to lock as the fixed configuration". **That conclusion was WRONG
and is withdrawn** -- not because of genuine pipeline determinism, but because the log-file discovery
bug (see the correction in markdown C.3 and README.md "Log-file discovery bug v3") made this cell fail
to find the zeus alert file in ALL 5 runs, not just some. It has been fixed and RE-verified with the
same file-discovery mechanism as C.3 -- result of the 28 August run: `[5697, 5697, 5697, 5697, 5697]`,
**genuinely identical**; the zeus determinism claim is now VALID.

**CORRECTION #2 (28 August 2026)**: the same 28 August run ALSO compared `tinba.pcap` (the
`[diag-v3]` data in Section C.3, run of 26 vs 28 August) and found `tinba` **NOT deterministic**
(`dropped_throttle`/`written` differing almost 2x even though the `total` of `log_event` calls was
identical). The root cause is separate from the above: the raw `tinba.pcap` timestamps are near-epoch
(1970-01-01), fail the Fix #2 sanity check, and fall back to the real wall clock -- see markdown
"C.1b (continued)" for the fix (a relative-timestamp fallback branch, tested in 11 scenarios + direct
confirmation with `pcapy` against the real `tinba.pcap`/`zeus.pcap`). The code cell below now checks
**zeus AND tinba** automatically in one go, not just zeus -- it **must be run AGAIN** with this fixed
notebook before the tinba determinism conclusion may be stated in the paper's methodology.

This cell is NOT part of the main pipeline (it does not affect `maltrail_results`/`all_maltrail` in
Sections C.3/C.4) -- its purpose is to verify whether the combination of the `PROCESS_COUNT 1` fix
(C.2) + the offline-timestamp patch (C.1b) really removes the non-determinism of the Zeus alert-count drift.

**History**: the first version of this cell (2 runs) was run and the results **DIFFERED (17 vs 18)** --
the `PROCESS_COUNT 1` fix alone was PROVEN INSUFFICIENT. Further investigation (reading the
`old/sensor.py` code directly + empirical tests with synthetic pcaps) found that the wall-clock
substitution (`time.time()`) in the capture loop itself is enough to be a source of drift, even with
`PROCESS_COUNT 1` removing multiprocessing entirely -- see cell C.1b above for the patch replacing it
with the original pcap timestamp (`header.getts()`) for offline mode only, with a safe fallback in
case it turns out not to be valid in the local environment.

**This cell now runs N times** (`DETERMINISM_CHECK_N_RUNS`, default 5) for **each key** in
`DETERMINISM_CHECK_KEYS` (default `["zeus", "tinba"]`), using the SAME trail snapshot (`--offline`)
in all runs per pcap. The result of each run is stored separately from the official
`data/maltrail_out/<key>.log` (prefix `_determinism_check_<key>_`), so it does not interfere with the
results of Sections C.3/C.4. Add other pcap keys to that list if new pcaps later also need checking.

**If the result is IDENTICAL across all N runs**: the patch + fix are proven sufficient -- safe to use
as the fixed configuration for all offline-replay experiments in this research; mention it in the
paper's methodology as a verified step (include the N runs & the verification date).

**If something STILL differs**: report all the numbers (not just "still different") -- including
whether the line `[!] PATCH offline-timestamp: ... falling back to the wall clock` appears in the C.1b/C.3
output above (if it does, `header.getts()` is really not valid in the local environment, and the
problem goes back to its starting point -- not a new bug). If that line does NOT appear (meaning the
patch is fully active) but the results still differ, there is a THIRD source of non-determinism not
yet found -- this needs a new investigation; do not assume it is automatically the same as the earlier problem.

In [ ]:
# --- C.5 -- Determinism verification: run Maltrail N times in a row (default 5) ---
# Fix of 28 August 2026: previously only "zeus" was checked here manually. Now two
# pcaps are checked automatically in one go: "zeus" (real timestamps, passed the Fix #2 sanity check
# from the start) and "tinba" (near-epoch timestamps, only just fixed via the
# offline-timestamp-relative-fallback patch above -- see markdown "C.1b (continued)"). The two
# represent two DIFFERENT classes of problem found so far; if a new pcap later also shows
# drift, add its key to the list below.
DETERMINISM_CHECK_KEYS = ["zeus", "tinba"]
DETERMINISM_CHECK_N_RUNS = 5       # raised from 2 -> 5 after the first 2 zeus runs turned out DIFFERENT (17 vs 18)

if MALTRAIL_DIR.exists() and SENSOR_PY.exists() and "MALTRAIL_PYTHON" in globals():
    _all_results = {}
    for _check_key in DETERMINISM_CHECK_KEYS:
        if _check_key not in downloaded:
            print(f"[{_check_key}] skipped -- not in `downloaded` (check Section B).")
            continue
        check_pcap = downloaded[_check_key]
        run_lines = []
        print(f"=== Determinism verification: {_check_key} ===")
        for i in range(1, DETERMINISM_CHECK_N_RUNS + 1):
            # Fix of 26 August 2026: the old `maltrail_dated_log_path()` was PROVEN WRONG for
            # zeus (see README.md "Log-file discovery bug v3") -- zeus.pcap is originally dated
            # 2012-05-29, not today, so it always reported "no log file" even though Maltrail HAD
            # written real alerts. The same discovery functions as C.3 are used
            # (snapshot_dated_logs/collect_new_log_lines, tested in 9 isolated scenarios).
            _before_snapshot = snapshot_dated_logs()
            proc = run_maltrail_on_pcap(MALTRAIL_PYTHON, check_pcap, offline=True)
            _new_bytes, _touched_files = collect_new_log_lines(_before_snapshot)
            if _new_bytes:
                dest = MALTRAIL_OUT_DIR / f"_determinism_check_{_check_key}_run{i}.log"
                dest.write_bytes(_new_bytes)
                n = _new_bytes.count(b"\n")
            else:
                dest, n = None, 0
            run_lines.append(n)
            print(f"[{_check_key} Run {i}/{DETERMINISM_CHECK_N_RUNS}] returncode={proc.returncode}, "
                  f"alert lines={n}"
                  + (f", log={dest.name} (from: {', '.join(_touched_files)})" if dest
                     else " (no dated log file grew -> 0 alerts)"))
            for _diag_line in proc.stdout.splitlines():
                if _diag_line.startswith("[diag"):  # catches [diag] (v1), [diag-v2], and [diag-v3]
                    print("   ", _diag_line)

        _all_results[_check_key] = run_lines
        print(f"[{_check_key}] all runs: {run_lines}")
        if len(set(run_lines)) == 1:
            print(f"[{_check_key}] RESULT: IDENTICAL across all {DETERMINISM_CHECK_N_RUNS} runs "
                  f"({run_lines[0]} lines) -> deterministic.")
        else:
            lo, hi = min(run_lines), max(run_lines)
            print(f"[{_check_key}] RESULT: STILL DIFFERENT (range {lo}-{hi} lines over "
                  f"{DETERMINISM_CHECK_N_RUNS} runs: {run_lines}) -> a source of "
                  "non-determinism remains; record the full run_lines figures for investigation.")
        print()

    print("=== Summary of all pcaps checked ===")
    _all_deterministic = all(len(set(v)) == 1 for v in _all_results.values()) if _all_results else False
    for k, v in _all_results.items():
        status = "DETERMINISTIC" if len(set(v)) == 1 else "STILL DIFFERENT"
        print(f"  {k}: {v} -> {status}")
    if _all_results and _all_deterministic:
        print()
        print("All pcaps checked are deterministic -> the PROCESS_COUNT 1 fix + offline-timestamp patch")
        print("(including the relative-timestamp fallback branch) are PROVEN SUFFICIENT for these pcaps.")
        print("Safe to lock as this experiment's fixed configuration -- mention it in the paper's methodology,")
        print("INCLUDING the list of verified pcaps (not a generalisation to all pcaps).")
else:
    print("Skipped: MALTRAIL_DIR/SENSOR_PY incomplete, or Section C.3 (previous cell) has not yet "
          "run successfully (MALTRAIL_PYTHON does not exist yet).")


## Section D — Zeek (flow monitoring)

This is the technically more certain part [HIGH CONFIDENCE — standard Zeek idiom].

### D.1 — Installation (if not present yet)

```bash
brew install zeek
zeek --version
```


In [ ]:
zeek_path = shutil.which("zeek")
if zeek_path is None:
    print("Zeek is not on PATH yet. Run `brew install zeek` first, then re-run this cell.")
else:
    version = subprocess.run([zeek_path, "--version"], capture_output=True, text=True)
    print("Zeek found:", zeek_path)
    print(version.stdout.strip())


In [ ]:
def run_zeek_on_pcap(pcap_path: Path, out_subdir: Path) -> subprocess.CompletedProcess:
    '''Run Zeek offline on one pcap. Zeek writes its logs to the CWD, so we cd into the output folder first.'''
    out_subdir.mkdir(parents=True, exist_ok=True)
    cmd = [zeek_path, "-C", "-r", str(pcap_path.resolve()), "local"]
    print("Running:", " ".join(cmd), "  (cwd:", out_subdir, ")")
    return subprocess.run(cmd, capture_output=True, text=True, cwd=out_subdir, timeout=300)


zeek_run_results = {}
if zeek_path is None:
    print("Zeek is not installed/not on PATH -> this cell is skipped.")
    print("Run `brew install zeek` first, then re-run this cell (only this cell, no need to start from the top).")
else:
    for key, pcap_path in downloaded.items():
        out_dir = ZEEK_OUT_DIR / key
        proc = run_zeek_on_pcap(pcap_path, out_dir)
        zeek_run_results[key] = proc
        conn_log = out_dir / "conn.log"
        status = "conn.log PRESENT" if conn_log.exists() else "conn.log MISSING — check stderr"
        print(f"[{key}] returncode={proc.returncode} -> {status}")
        if proc.returncode != 0:
            print("   stderr:", proc.stderr.strip()[:500])


### D.2 — Parse `conn.log` into pandas

Standard Zeek TSV format: the `#fields` line defines the column names, the `#types` line defines the data types, other lines are data (tab-separated, `-` = empty value).


In [ ]:
import pandas as pd

def parse_zeek_log(log_path: Path) -> pd.DataFrame:
    fields, rows = None, []
    with open(log_path) as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith("#fields"):
                fields = line.split("\t")[1:]
            elif line.startswith("#") or not line.strip():
                continue
            else:
                rows.append(line.split("\t"))
    if fields is None:
        raise ValueError(f"No #fields line found in {log_path} — the format is not as expected.")
    df = pd.DataFrame(rows, columns=fields)
    return df.replace("-", pd.NA)


conn_frames = []
for key, pcap_path in downloaded.items():
    conn_log = ZEEK_OUT_DIR / key / "conn.log"
    if not conn_log.exists():
        print(f"[{key}] no conn.log, skipped")
        continue
    df = parse_zeek_log(conn_log)
    df["pilot_key"] = key
    # BUG (found in a real run on 25 August 2026): this previously searched PILOT_MANIFEST only,
    # crashing with StopIteration as soon as a key from EXTENDED_MANIFEST (e.g. iot23_capture1) got here,
    # because it was never found in PILOT_MANIFEST. Fixed to ALL_MANIFEST.
    ground_truth = next(e.ground_truth for e in ALL_MANIFEST if e.key == key)
    df["ground_truth"] = ground_truth
    conn_frames.append(df)
    print(f"[{key}] {len(df)} flow rows, ground_truth={ground_truth}")

if conn_frames:
    all_conn = pd.concat(conn_frames, ignore_index=True)
    print()
    print("Total combined flow rows:", len(all_conn))
    display(all_conn.head())
else:
    all_conn = pd.DataFrame()
    print("No conn.log has been parsed yet.")


## Section E — Pilot summary & next steps

This table combines the ground truth, the number of Zeek flow records, and the number of Maltrail alerts (raw lines + unique observables) per pcap.

**Results of the 5-pcap pilot obtained earlier (24 August 2026, run manually in a terminal before this recipe was embedded in the notebook)** — if this notebook is re-run from the start, the `summary_df` table below should produce the same pattern:

| pcap | ground truth | Maltrail result |
|---|---|---|
| Zeus | malware | ALERT — `"blackshades (malware)"` + several heuristics |
| Tinba | malware | ALERT — 500+ `.xyz` DGA domains, all `(static)` |
| Dridex | malware | 0 alerts — possibly IOC staleness (2016 pcap vs 2026 blacklist), NEEDS FURTHER VERIFICATION |
| BitTorrent | benign | 0 alerts (good for specificity) |
| Skype | benign | 0 alerts (good for specificity) |

Important methodological note: the label assigned by Maltrail (e.g. `"blackshades"` for a pcap named `Zeus.pcap`) **does not always match the family name in the dataset** — blacklist-based detection matches known indicators (domains/IPs), not file names. The correct evaluation is a **binary hit/miss** (whether Maltrail raises any alert at all), not an exact family-name match.


In [ ]:
summary_rows = []
for entry in ALL_MANIFEST:   # <- extended 25 August 2026, previously PILOT_MANIFEST only
    key = entry.key
    n_flows = int((all_conn["pilot_key"] == key).sum()) if not all_conn.empty else None
    m = maltrail_summary.get(key, {})
    summary_rows.append({
        "pcap": key,
        "source_dataset": entry.source_dataset,
        "family": entry.family,
        "ground_truth": entry.ground_truth,
        "n_zeek_flows": n_flows,
        "n_maltrail_alert_lines": m.get("n_alert_lines"),
        "n_maltrail_unique_observables": m.get("n_unique_observables"),
        "maltrail_hit": bool(m.get("n_alert_lines")) if m.get("n_alert_lines") is not None else None,
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

summary_path = RESULTS_DIR / "pilot_summary_layer1.csv"
summary_df.to_csv(summary_path, index=False)
print("Saved to:", summary_path)


### Checklist after this pilot runs

- [x] Confirm the `old/sensor.py` CLI flags for offline pcaps — `-r`, `-c`, `--console`, `--offline` (Sections C.1-C.3).
- [x] Confirm the location & format of the Maltrail log — `LOG_DIR/<YYYY-MM-DD>.log`, regex parser in Section C.4.
- [x] `n_maltrail_alerts` filled in automatically in the Section E summary.
- [x] Sanity check: Zeus & Tinba (malware) raise alerts; BitTorrent & Skype (benign) do not — **except Dridex** (malware, 0 alerts), which needs an explicit explanation in the paper, not to be hidden.
- [ ] Investigate the IOC-staleness hypothesis for the 0-alert Dridex result (e.g. check manually whether the IOCs of this 2016 capture have ever been in any trail, or cross-check with the Zeek `conn.log` of the same pcap for suspicious patterns that slip past the blacklist).
- [ ] Once this 5-file pilot makes sense: run Section B.2 (Cridex + Shifu, optional) to extend to 7 pcaps.
- [ ] Find & verify an Ursnif pcap (not yet in the manifest — empty placeholder).
- [ ] Decide the status of the `Zbot`/`Gootkit` label (Botnet-224-1) — used with an uncertainty qualification, or removed from the corpus.
- [ ] Once the corpus is considered sufficient: `all_conn` (the Zeek flow dataframe) becomes the starting point of Section B of the Layer 2 experiment — extraction of 6 candidate behaviour features (bytes in/out ratio, port entropy, inter-arrival-time regularity, etc. — see the PIVOT section of the project notes).


## Section F — CTU-13 (groundwork for a coherent population for Layer 2) — NOT YET IMPLEMENTED

CTU-13 (Stratosphere Lab) is structured differently from `EXTENDED_MANIFEST` above, so it is deliberately NOT
forced into the one-pcap-one-entry `DatasetEntry` pattern:

- Distributed as a single bundle `CTU-13-Dataset.tar.bz2` (1.9 GB, CC-BY 2.0 licence), not as separate
  pcap files per scenario — it must be downloaded+extracted before its per-scenario content is known.
- Each scenario (Botnet-42 to Botnet-54) is NOT a single pcap: there is a FULL pcap (untruncated)
  for the botnet portion of the traffic only, AND a labelled NetFlow file (Argus,
  bidirectional) for ALL the scenario traffic (Normal+Botnet+Background together) — two sources of
  truth at different levels (pcap vs flow), not one file.
- A sensible order of work if this is continued: (1) download the bundle manually on the local machine
  (mcfp.felk.cvut.cz was blocked by the network of the drafting environment, see the note in the download
  cell above) → put it in `data/ctu13_raw/` → (2) extract, pick the 1-2 smallest scenarios first (Table 3 in
  Garcia et al. 2014 should list the size per scenario, not yet read in detail) → (3) the botnet pcap
  can go straight through Maltrail (same pattern as Section C above, just add a manual entry to the
  `downloaded` dict) → (4) its labelled NetFlow is used separately for Layer 2 (not Zeek
  conn.log, so the Section D.2 parser does NOT fit directly — Argus fields differ from Zeek's, a new
  parser is needed if it is to be used).

Bundle URL (verified via a directory listing on 25 August 2026):
`https://mcfp.felk.cvut.cz/publicDatasets/CTU-13-Dataset/CTU-13-Dataset.tar.bz2`

This is only a groundwork note so it is not lost — the code implementation follows once it is
decided whether to continue on this path (see the discussion of per-host ground truth & a coherent
population for Layer 2).
